# Customer Review Theme Analysis

## Objective

This analysis explores recurring themes in written customer reviews, with a primary focus on low-rated reviews.

The purpose is to provide qualitative context for the earlier quantitative analysis of customer satisfaction and delivery performance.

This is not a sentiment analysis or machine learning project. Instead, the analysis uses a transparent and targeted approach to examine commonly occurring themes in customer feedback.

The analysis does not establish the causes of low review scores. Review themes represent issues reported by customers and should be interpreted as qualitative evidence that may provide additional context for the quantitative findings.

In [1]:
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect("../data/olist_ecommerce.db")

print("SQLite database connection created successfully.")

SQLite database connection created successfully.


## 1. Data Preparation and Review Coverage

This section examines the availability of written customer reviews and identifies the subset of reviews that can be used for qualitative theme analysis.

In [3]:
tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", conn)

tables

,name
0,category_translation
1,customers
2,order_items
3,orders
4,payments
5,products
6,reviews
7,sellers


In [4]:
reviews_df = pd.read_sql_query(
    "SELECT * FROM reviews LIMIT 5;",
    conn
)

reviews_df

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [5]:
query = """
SELECT
    COUNT(*) AS total_review_records,

    SUM(
        CASE
            WHEN review_comment_message IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS reviews_with_message,

    SUM(
        CASE
            WHEN review_comment_title IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS reviews_with_title,

    SUM(
        CASE
            WHEN review_comment_message IS NOT NULL
              OR review_comment_title IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS reviews_with_any_text

FROM reviews;
"""

df_review_coverage = pd.read_sql_query(query, conn)

df_review_coverage

,total_review_records,reviews_with_message,reviews_with_title,reviews_with_any_text
0,99224,40977,11568,42706


### Review Text Coverage

The reviews table contains 99,224 review records.

Of these records:

- 40,977 contain a written review message.
- 11,568 contain a review title.
- 42,706 contain either a written title or message.

Therefore, approximately 43% of review records contain some form of written customer feedback.

The remaining review records contain only a numerical review score.

As a result, the qualitative theme analysis is limited to customers who provided written feedback and should not be interpreted as representing the experiences or opinions of all customers in the dataset.

In [6]:
query = """
SELECT
    review_score,

    COUNT(*) AS total_reviews,

    SUM(
        CASE
            WHEN review_comment_message IS NOT NULL
              OR review_comment_title IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS reviews_with_text

FROM reviews

WHERE review_score IN (1, 2)

GROUP BY review_score

ORDER BY review_score;
"""

df_low_rating_coverage = pd.read_sql_query(query, conn)

df_low_rating_coverage

,review_score,total_reviews,reviews_with_text
0,1,11424,8829
1,2,3151,2165


In [7]:
query = """
SELECT
    review_score,
    review_comment_title,
    review_comment_message
FROM reviews
WHERE review_score IN (1, 2)
  AND review_comment_message IS NOT NULL
ORDER BY RANDOM()
LIMIT 20;
"""

df_low_reviews_sample = pd.read_sql_query(query, conn)

df_low_reviews_sample

,review_score,review_comment_title,review_comment_message
0,2,None,A base da cadeira veio quebrada. Podem me mand...
1,1,None,"Eu comprei dois kits cobre leito, só chegou um"
2,2,None,O produto ainda não foi entregue e o prazo já ...
3,1,Produto não entregue,O produto ainda não foi entregue. Recebi a men...
4,1,None,Ainda não recomendaria pq não recebi o produto
5,1,None,Produto ainda não chegou
6,1,None,Cancelei o pedido.
7,1,None,meu produto não chego ainda y já paso o praso ...
8,1,None,"Estou com meu pedido desde 15/02, ainda não re..."
9,1,None,Compreis os 2 itens só recebi a película de vi...


## Sampling Low-Rated Reviews for Exploratory Theme Analysis

A targeted sample of written 1-star and 2-star reviews is extracted for qualitative examination.

The purpose is to identify recurring customer issues and provide context for the quantitative findings from the previous analysis.

Because this is not a full NLP or manually coded dataset, the identified themes should be interpreted as exploratory patterns observed within the sampled reviews rather than exact proportions of all customer complaints.

In [8]:
conn = sqlite3.connect("../data/olist_ecommerce.db")

In [9]:
pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name;
""", conn)

,name
0,category_translation
1,customers
2,order_items
3,orders
4,payments
5,products
6,reviews
7,sellers


In [10]:
df_review_sample = pd.read_sql_query("""
SELECT
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message
FROM reviews
WHERE review_score IN (1, 2)
AND (
    review_comment_title IS NOT NULL
    OR review_comment_message IS NOT NULL
)
ORDER BY review_score, RANDOM()
LIMIT 100;
""", conn)

df_review_sample

,review_id,order_id,review_score,review_comment_title,review_comment_message
0,9b27ecb1f3f5cd5f982f4b03f383e301,b155e845bb514ad16d9cb7712ef9be90,1,Ainda não recebi o produt,Ainda esta no prazo para entrega
1,9d8c414738305af1419d9ab505caceef,b3666570c67eca46e04bc11a7285f0db,1,Produto de qualidade ruim,"Produto completamente diferente da propaganda,..."
2,4a317fd181969db85d0cd7353d17150f,bf80d30d4654a34a647d0e5db84a5e2e,1,None,"comprei o cartucho canon como original, pela e..."
3,a48da15d6d538d78c6d7a54cdec219eb,c97cbfce4294d32b1566a3132cbf4127,1,None,Comprei duas luminárias iguais. Chegou uma cai...
4,4864c7a3d086f33fa4c38ca2bb63b441,1b025e1c2a64571df27e2229bcfe74dd,1,None,O PRODUTO DESCOLOU E ATÉ AGORA NAO RESPONDERAM...
...,...,...,...,...,...
95,eed7eaa5642e6e0f1ba87f8826860579,20bc33ea85750fd7d20d7b72c5c4b36d,1,None,O produto veio rasgado
96,b152c3916fac087c4eba4a98ac7a5b95,bcc93ae5358fc8dbcbcb60fbcfff8f41,1,None,"veio fora do prazo, demorou mais de um mês pra..."
97,b5fb1dd1967827e894ed62b6f9096264,20da65274d671907282e06147c655ae4,1,None,não chegou ainda
98,559f7b4591b774c0d32dd2c9dd9cb18d,56abbae9014467d35f46261ee6f7fa71,1,Não recebi o produto.,"Infelizmente, o produto não chegou até o momen..."


In [11]:
df_review_sample["review_text"] = (
    df_review_sample["review_comment_title"].fillna("") + " " +
    df_review_sample["review_comment_message"].fillna("")
)

df_review_sample[[
    "review_score",
    "review_text"
]].head(10)

,review_score,review_text
0,1,Ainda não recebi o produt Ainda esta no prazo ...
1,1,Produto de qualidade ruim Produto completament...
2,1,"comprei o cartucho canon como original, pela ..."
3,1,Comprei duas luminárias iguais. Chegou uma ca...
4,1,O PRODUTO DESCOLOU E ATÉ AGORA NAO RESPONDERA...
5,1,"Horrivel o atendimento Fiz a compra, o produto..."
6,1,Não gostei da
7,1,Passou dos dias previstos.
8,1,De acordo com meus pedidos Ainda não recebi a...
9,1,Recebi produto diferente do que comprei.\r\nA...


In [12]:
pd.set_option("display.max_colwidth", None)

df_review_sample[[
    "review_score",
    "review_text"
]].head(20)

,review_score,review_text
0,1,Ainda não recebi o produt Ainda esta no prazo para entrega
1,1,"Produto de qualidade ruim Produto completamente diferente da propaganda,de péssima qualidade,os desenhos da cortina é muito claro e manchado,na foto as cores são muito mais vibrantes,só vou ficar com o produto pq sou obrigada."
2,1,"comprei o cartucho canon como original, pela embalagem e por falta de informação de que era compatível, e recebi um cartucho compatível, que ao que tudo indica é recarragado. Não era isso que queria."
3,1,"Comprei duas luminárias iguais. Chegou uma caixa grande contendo somente uma luminária, a NF descrevendo 2 luminárias e o status de entrega como se tivesse entregado. Um trabalho danado resolver."
4,1,O PRODUTO DESCOLOU E ATÉ AGORA NAO RESPONDERAM E UMA VERGONHA
5,1,"Horrivel o atendimento Fiz a compra, o produto não foi entregue,a nota foi emitida dia 21/05/18 a entrega era para o dia 15/05/18, estou aguardando reembolso, não recomendo a loja, horrível, nunca mais comprarei com eles !"
6,1,Não gostei da
7,1,Passou dos dias previstos.
8,1,"De acordo com meus pedidos Ainda não recebi as Toalhas de Jogo De Banho Florença Bordado 5 Peças 100% Algodão Branco, recebi apenas Kit Jogo De Cama Florença Queen Com Toalhas De Banho 9 Peças"
9,1,Recebi produto diferente do que comprei.\r\nAbri reclamação tem 05 dias e continuo aguardando solução


## Exploratory Theme Identification

The sampled reviews are examined for recurring issues reported by customers. Based on an initial review of the written feedback, several broad themes are identified, including delivery problems, product quality issues, incorrect or incomplete orders, customer service concerns, and refund or cancellation issues.

Theme identification is exploratory and keyword-based. A review may contain more than one theme, and the results should not be interpreted as exact proportions of all customer complaints.

In [13]:
import re

theme_patterns = {
    
    "Delivery / Product Not Received": [
        r"não recebi",
        r"nao recebi",
        r"não chegou",
        r"nao chegou",
        r"ainda não recebi",
        r"ainda nao recebi",
        r"atraso",
        r"atrasada",
        r"esperando"
    ],
    
    "Product Quality / Defect": [
        r"defeito",
        r"defeituoso",
        r"péssima qualidade",
        r"pessima qualidade",
        r"produto ruim",
        r"quebrado",
        r"rachad",
        r"usado"
    ],
    
    "Wrong / Different Product": [
        r"produto errado",
        r"enviaram outro",
        r"enviado errado",
        r"diferente do escolhido",
        r"produto diferente"
    ],
    
    "Incomplete / Missing Items": [
        r"faltando",
        r"faltou",
        r"veio faltando",
        r"produto incompleto",
        r"só recebi",
        r"so recebi"
    ],
    
    "Customer Service / Communication": [
        r"sem resposta",
        r"não consegui resposta",
        r"nao consegui resposta",
        r"não entra em contato",
        r"nao entra em contato",
        r"ninguém responde",
        r"ninguem responde"
    ],
    
    "Refund / Cancellation": [
        r"dinheiro de volta",
        r"cancelamento",
        r"cancelado",
        r"reembolso"
    ]
}

print("Theme patterns defined successfully.")

Theme patterns defined successfully.


In [14]:
def matches_theme(text, patterns):
    text = str(text).lower()
    
    return any(
        re.search(pattern, text)
        for pattern in patterns
    )


theme_results = {}

for theme, patterns in theme_patterns.items():
    
    theme_results[theme] = df_review_sample[
        "review_text"
    ].apply(
        lambda text: matches_theme(text, patterns)
    ).sum()


df_theme_counts = (
    pd.DataFrame(
        list(theme_results.items()),
        columns=["theme", "matched_reviews"]
    )
    .sort_values(
        "matched_reviews",
        ascending=False
    )
    .reset_index(drop=True)
)

df_theme_counts

,theme,matched_reviews
0,Delivery / Product Not Received,25
1,Product Quality / Defect,7
2,Incomplete / Missing Items,7
3,Refund / Cancellation,5
4,Customer Service / Communication,2
5,Wrong / Different Product,1


### Theme Validation

The keyword-screening results are manually inspected to verify that the matched reviews are relevant to each identified theme.

This validation step helps ensure that keyword matches are interpreted as meaningful customer issues rather than relying solely on automated keyword detection.

In [15]:
delivery_patterns = theme_patterns["Delivery / Product Not Received"]

df_delivery_matches = df_review_sample[
    df_review_sample["review_text"].apply(
        lambda text: matches_theme(text, delivery_patterns)
    )
][["review_score", "review_text"]]

df_delivery_matches

,review_score,review_text
0,1,Ainda não recebi o produt Ainda esta no prazo para entrega
8,1,"De acordo com meus pedidos Ainda não recebi as Toalhas de Jogo De Banho Florença Bordado 5 Peças 100% Algodão Branco, recebi apenas Kit Jogo De Cama Florença Queen Com Toalhas De Banho 9 Peças"
13,1,Faz 23 dias da minha compra e até o momento não recebi o meu produto .
16,1,nao sei como posso avaliar se nao recebi a mercadoria
17,1,Nao chegou produto So chegou um potecde creme faltou um pote
20,1,Sempre comprei pela lannister.com e essa foi a única vez em que o prazo encerrou e ainda não recebi o produto. Inclusive comprei por outras duas lojas bem depois de vocês e os produtos já chegaram.
25,1,ainda aguardo o envio do meu produto. Está em atraso.
28,1,"Até agora não recebi meu produto, estou esperando pois se não for enviado quero a devolução do meu dinheiro"
32,1,"Ainda não recebi o produto, mas acredito que a demora seja por conta da greve dos caminhoneiros. Então, não posso opinar sobre a qualidade dos serviços."
33,1,O prazo de entrega expirou em 28/12/17 e até o momento não recebi meu produto.


In [16]:
validation_results = []

for theme, patterns in theme_patterns.items():
    
    if theme == "Delivery / Product Not Received":
        continue
    
    matches = df_review_sample[
        df_review_sample["review_text"].apply(
            lambda text: matches_theme(text, patterns)
        )
    ][["review_score", "review_text"]].copy()
    
    matches["theme"] = theme
    
    validation_results.append(matches)


df_other_theme_matches = pd.concat(
    validation_results,
    ignore_index=True
)

df_other_theme_matches[
    ["theme", "review_score", "review_text"]
]

,theme,review_score,review_text
0,Product Quality / Defect,1,"Produto de qualidade ruim Produto completamente diferente da propaganda,de péssima qualidade,os desenhos da cortina é muito claro e manchado,na foto as cores são muito mais vibrantes,só vou ficar com o produto pq sou obrigada."
1,Product Quality / Defect,1,"Produto entregue não condiz com o produto comprado, péssima qualidade, assessório veio quebrado."
2,Product Quality / Defect,1,Produto com defeito!
3,Product Quality / Defect,1,"A mesa tem defeitos em sua tabua, e as pernas parecem que foram pintadas de ultima hora, as duas tabuas que ficam entre as pernas possuem tamanhos diferentes. Faltam parafusos, vieram 8 de 16."
4,Product Quality / Defect,1,Produto com defeito Eu solicitei devolucao do valor pois o produto veio com defeito e nao foi aceito pela impressora apontando defeito.
5,Product Quality / Defect,1,O peoduto veio com defeito
6,Product Quality / Defect,1,O produto está com defeito e estou aguardando a loja entrar em contato comigo este tempo todo e até hoje nada...
7,Wrong / Different Product,1,Recebi produto diferente do que comprei.\r\nAbri reclamação tem 05 dias e continuo aguardando solução
8,Incomplete / Missing Items,1,Nao chegou produto So chegou um potecde creme faltou um pote
9,Incomplete / Missing Items,1,"Veio faltando um rolo Veio faltando um rolo de papel de parede. Como vou ter neném na semana que vem, vou querer meu dinheiro de volta, pois não vai dar tempo de chegar outro rolo caso vcs mandem. Eu vou devolver esse"


## Exploratory Theme Summary

The validated keyword-screening results are summarized below to show the frequency of broad customer issues identified within the sample of 100 low-rated written reviews.

Because themes are not mutually exclusive, a single review may be associated with more than one theme. These frequencies describe patterns observed within the exploratory sample and should not be interpreted as exact proportions of all low-rated reviews.

In [17]:
df_theme_summary = df_theme_counts.copy()

sample_size = len(df_review_sample)

df_theme_summary["percentage_of_sample"] = (
    df_theme_summary["matched_reviews"] / sample_size * 100
).round(1)

df_theme_summary

,theme,matched_reviews,percentage_of_sample
0,Delivery / Product Not Received,25,25.0
1,Product Quality / Defect,7,7.0
2,Incomplete / Missing Items,7,7.0
3,Refund / Cancellation,5,5.0
4,Customer Service / Communication,2,2.0
5,Wrong / Different Product,1,1.0


## Key Findings

The exploratory analysis of 100 written 1-star and 2-star reviews identified several recurring customer issues.

### 1. Delivery and Product Non-Receipt

Delivery and product non-receipt issues were the most frequently identified theme, appearing in 24 of the 100 sampled reviews. Customers commonly reported delayed deliveries, products not arriving, and orders marked as delivered despite the customer not receiving them.

This qualitative finding provides additional context for the earlier quantitative analysis, which identified a strong association between delivery delays and lower review scores.

### 2. Product and Order Fulfilment Issues

Other recurring issues included incomplete or missing items, product quality or defects, and receiving products that differed from what was expected or ordered.

These findings suggest that customer experience concerns are not limited to delivery performance and may also arise from product quality and order fulfilment problems.

### 3. Communication and Resolution Concerns

Some reviews described difficulties obtaining information or responses from sellers or the platform. Communication issues often appeared alongside delivery or order-related problems.

Cancellation and refund-related concerns also appeared in a small number of reviews. These should be interpreted cautiously, as they may represent consequences of other customer experience problems rather than independent drivers of dissatisfaction.

## Interpretation and Limitations

This analysis is exploratory and based on a sample of 100 written low-rated reviews.

The keyword-based screening does not represent a complete classification of all customer complaints, and the identified theme frequencies should not be interpreted as exact proportions of all low-rated reviews.

Reviews may also contain multiple issues and can therefore be associated with more than one theme.

Nevertheless, the recurring themes provide qualitative context that complements the quantitative analysis conducted in earlier phases.

In [18]:
# Restore cleaned reviews dataset

reviews_clean = pd.read_csv("../data/cleaned/reviews_clean.csv")

print("reviews_clean shape:", reviews_clean.shape)
display(reviews_clean.head())

reviews_clean shape: (99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53


In [19]:
# Phase 6B — Prepare customer reviews for AI/NLP analysis

reviews_text = reviews_clean[
    [
        "order_id",
        "review_id",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    ]
].copy()

# Combine title and message into one review text field
reviews_text["review_text"] = (
    reviews_text["review_comment_title"].fillna("").astype(str).str.strip()
    + " "
    + reviews_text["review_comment_message"].fillna("").astype(str).str.strip()
).str.strip()

# Flag reviews containing usable text
reviews_text["has_review_text"] = reviews_text["review_text"].ne("")

print("Total review records:", len(reviews_text))
print("Reviews with usable text:", reviews_text["has_review_text"].sum())
print("Reviews without usable text:", (~reviews_text["has_review_text"]).sum())

display(
    reviews_text[
        reviews_text["has_review_text"]
    ][
        ["order_id", "review_score", "review_text"]
    ].head(10)
)

Total review records: 99224
Reviews with usable text: 42687
Reviews without usable text: 56537


,order_id,review_score,review_text
3,658677c97b385a9be170737859d3511b,5,Recebi bem antes do prazo estipulado.
4,8e6bfb81e283fa7e4f11123a3fb894f1,5,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa
9,b9bf720beb4ab3728760088589c62129,4,recomendo aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho
12,9d6f15f95d01e79bd1349cc208361f09,4,"Mas um pouco ,travando...pelo valor ta Boa."
15,e51478e7e277a83743b6f9991dbfa3fb,5,"Super recomendo Vendedor confiável, produto ok e entrega antes do prazo."
16,0dacf04c5ad59fd5a0cc1faa07c34e39,2,"GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E ESSA COMPRA AGORA ME DECPCIONOU"
19,583174fbe37d3d5f0d6661be3aad1786,1,Não chegou meu produto Péssimo
22,4fc44d78867142c627497b60a7e0228a,5,Ótimo Loja nota 10
24,79832b7cb59ac6f887088ffd686e1d5e,5,obrigado pela atençao amim dispensada
27,2ca73e2ff9e3a186ad1e1ffb9b1d9c10,5,"A compra foi realizada facilmente.\r\nA entrega foi efetuada muito antes do prazo dado.\r\nO produto já começou a ser usado e até o presente,\r\nsem problemas."


In [20]:
# Phase 6B — Review text profiling

review_text_data = reviews_text[
    reviews_text["has_review_text"]
].copy()

# Text length metrics
review_text_data["character_count"] = (
    review_text_data["review_text"].str.len()
)

review_text_data["word_count"] = (
    review_text_data["review_text"]
    .str.split()
    .str.len()
)

print("Written reviews:", len(review_text_data))

print("\nCharacter count:")
display(review_text_data["character_count"].describe())

print("\nWord count:")
display(review_text_data["word_count"].describe())

print("\nReview score distribution among written reviews:")
display(
    review_text_data["review_score"]
    .value_counts()
    .sort_index()
    .rename_axis("review_score")
    .reset_index(name="review_count")
)

print("\nShortest written reviews:")
display(
    review_text_data
    .sort_values("character_count")
    [["review_score", "review_text", "character_count", "word_count"]]
    .head(10)
)

Written reviews: 42687

Character count:


count    42687.000000
mean        69.035514
std         54.922285
min          1.000000
25%         27.000000
50%         53.000000
75%         95.000000
max        229.000000
Name: character_count, dtype: float64


Word count:


count    42687.000000
mean        11.718650
std          9.688165
min          1.000000
25%          4.000000
50%          9.000000
75%         16.000000
max         48.000000
Name: word_count, dtype: float64


Review score distribution among written reviews:


,review_score,review_count
0,1,8829
1,2,2165
2,3,3643
3,4,6274
4,5,21776



Shortest written reviews:


,review_score,review_text,character_count,word_count
49002,4,8,1,1
4575,1,8,1,1
62778,3,3,1,1
27041,1,.,1,1
34584,4,?,1,1
42131,5,.,1,1
598,4,4,1,1
65441,5,9,1,1
28716,4,7,1,1
583,4,5,1,1


In [21]:
# Phase 6B — Create NLP-ready review dataset

# Keep reviews containing at least 2 alphabetic characters.
# This removes entries such as ".", "?", "4", etc.
alphabetic_chars = (
    review_text_data["review_text"]
    .str.count(r"[A-Za-zÀ-ÿ]")
)

nlp_reviews = review_text_data[
    alphabetic_chars >= 2
].copy()

print("Reviews before text-quality filtering:", len(review_text_data))
print("Reviews retained for NLP:", len(nlp_reviews))
print("Reviews removed as non-informative:", 
      len(review_text_data) - len(nlp_reviews))

print("\nScore distribution after filtering:")

display(
    nlp_reviews["review_score"]
    .value_counts()
    .sort_index()
    .rename_axis("review_score")
    .reset_index(name="review_count")
)

print("\nShortest retained reviews:")

display(
    nlp_reviews
    .sort_values("character_count")
    [["review_score", "review_text", "character_count", "word_count"]]
    .head(10)
)

Reviews before text-quality filtering: 42687
Reviews retained for NLP: 42435
Reviews removed as non-informative: 252

Score distribution after filtering:


,review_score,review_count
0,1,8813
1,2,2161
2,3,3625
3,4,6219
4,5,21617



Shortest retained reviews:


,review_score,review_text,character_count,word_count
33141,3,ok,2,1
21239,5,Ok,2,1
52337,3,ok,2,1
29187,5,ok,2,1
28509,4,ok,2,1
55819,5,Ot,2,1
71797,4,Ok,2,1
41542,4,ok,2,1
27149,4,Ok,2,1
25405,3,Ok,2,1


In [22]:
# Phase 6B — Final text-quality filtering

alphabetic_chars = (
    nlp_reviews["review_text"]
    .str.count(r"[A-Za-zÀ-ÿ]")
)

nlp_reviews = nlp_reviews[
    alphabetic_chars >= 3
].copy()

print("Final NLP-ready reviews:", len(nlp_reviews))

print("Reviews removed by final text-quality rule:",
      len(review_text_data) - len(nlp_reviews))

print("\nShortest retained reviews:")

display(
    nlp_reviews
    .sort_values("character_count")
    [["review_score", "review_text", "character_count", "word_count"]]
    .head(10)
)

Final NLP-ready reviews: 42288
Reviews removed by final text-quality rule: 399

Shortest retained reviews:


,review_score,review_text,character_count,word_count
93616,4,Bom,3,1
6415,5,bom,3,1
33207,5,Top,3,1
70503,5,Bom,3,1
86395,4,Bom,3,1
30482,3,Boa,3,1
40048,2,Bom,3,1
70744,5,Bom,3,1
6412,4,boa,3,1
47704,5,Bom,3,1


In [23]:
# Phase 6B — Inspect review language and text composition

import re

def count_latin_chars(text):
    return len(re.findall(r"[A-Za-zÀ-ÿ]", str(text)))

def count_non_ascii_chars(text):
    return len(re.findall(r"[^\x00-\x7F]", str(text)))

nlp_reviews["latin_char_count"] = nlp_reviews["review_text"].apply(count_latin_chars)
nlp_reviews["non_ascii_char_count"] = nlp_reviews["review_text"].apply(count_non_ascii_chars)

print("Total NLP-ready reviews:", len(nlp_reviews))

print("\nReviews containing accented/non-ASCII characters:")
print(
    (nlp_reviews["non_ascii_char_count"] > 0).sum()
)

print("\nSample reviews:")
display(
    nlp_reviews[
        ["review_score", "review_text"]
    ].sample(15, random_state=42)
)

Total NLP-ready reviews: 42288

Reviews containing accented/non-ASCII characters:
23632

Sample reviews:


,review_score,review_text
47249,1,"Na Minha opinião , deveriam cumprir com prazo de entrega , era pra ser entregue até dia 12/05 e até hoje nada , se não estão disposto a cumprir com o combinado apenas , não determinem um prazo!"
22344,5,Ótimo e prazo rapido Preços ótimos
6008,4,O produto chegou antes do esperado e em perfeito estado! Recomendo muito.
3197,1,"O produto não foi entregue, tento falar com a empresa lannister e targaryene e só da instabilidade de página! Nem mesmo consigo cancelar minha compra! Total desrespeito com o cliente!"
91206,5,"O stark é show para compras na internet, fácil de comprar, tem meios seguros para efetuar o pagamento, é rápido na entrega.\r\n\r\nO único problema é esse programa de avaliação do vendedor que fica t"
21414,1,"O pedido foi entregue incompleto. Comprei duas capas para o celular moto g5 e duas películas de vidro para o mesmo celular, porém foram entregues somente uma capa e uma película."
29126,4,"Entrega incompleta Recebi parcialmente o pedido, veio as duas luminarias pendentes. Faltou o lustre aramado, entrei em contato aguardando retorno de uma posição de entrega do produto."
76678,5,Lindo produto e de qualidade. Amei.
86048,1,Produto veio com defeito
38159,1,aguardando produto. Peço por gentileza que vcs agilize a entrega deste produto.


In [24]:
# Phase 6B — Check NLP environment

import importlib.util

libraries = [
    "transformers",
    "torch",
    "sklearn",
    "nltk",
    "spacy"
]

for library in libraries:
    installed = importlib.util.find_spec(library) is not None
    print(f"{library}: {'Available' if installed else 'Not installed'}")

transformers: Available
torch: Available
sklearn: Available
nltk: Available
spacy: Not installed


In [25]:
# Phase 6B — Numerical review-score sentiment baseline

def score_to_sentiment(score):
    if score <= 2:
        return "Negative"
    elif score == 3:
        return "Neutral"
    else:
        return "Positive"

nlp_reviews["score_sentiment"] = (
    nlp_reviews["review_score"]
    .apply(score_to_sentiment)
)

sentiment_summary = (
    nlp_reviews["score_sentiment"]
    .value_counts()
    .reindex(["Negative", "Neutral", "Positive"])
    .fillna(0)
    .astype(int)
    .reset_index()
)

sentiment_summary.columns = [
    "sentiment",
    "review_count"
]

sentiment_summary["percentage"] = (
    sentiment_summary["review_count"]
    / len(nlp_reviews)
    * 100
).round(2)

display(sentiment_summary)

,sentiment,review_count,percentage
0,Negative,10969,25.94
1,Neutral,3611,8.54
2,Positive,27708,65.52


In [26]:
%pip install transformers torch sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [27]:
import transformers
import torch
import sentencepiece

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("SentencePiece:", sentencepiece.__version__)

Transformers: 5.17.0
PyTorch: 2.14.0+cpu
SentencePiece: 0.2.2


In [28]:
# Phase 6B — Test Portuguese sentiment model

from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_name = "pysentimiento/bertweet-pt-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

sentiment_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=-1
)

test_reviews = [
    "O produto chegou antes do prazo e em perfeito estado.",
    "O produto veio com defeito e estou muito decepcionado.",
    "O pedido foi entregue incompleto.",
    "Ótimo produto, recomendo.",
    "A entrega foi normal."
]

results = sentiment_pipeline(
    test_reviews,
    truncation=True
)

for text, result in zip(test_reviews, results):
    print(f"Review: {text}")
    print(f"Sentiment: {result['label']}")
    print(f"Confidence: {result['score']:.4f}")
    print("-" * 70)

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Review: O produto chegou antes do prazo e em perfeito estado.
Sentiment: NEU
Confidence: 0.7010
----------------------------------------------------------------------
Review: O produto veio com defeito e estou muito decepcionado.
Sentiment: NEG
Confidence: 0.9888
----------------------------------------------------------------------
Review: O pedido foi entregue incompleto.
Sentiment: NEU
Confidence: 0.7840
----------------------------------------------------------------------
Review: Ótimo produto, recomendo.
Sentiment: POS
Confidence: 0.9890
----------------------------------------------------------------------
Review: A entrega foi normal.
Sentiment: NEU
Confidence: 0.8809
----------------------------------------------------------------------


In [29]:
# Phase 6B — Create a human-validation sample

import pandas as pd

# Reload the review dataset after the kernel restart
reviews_clean = pd.read_csv("../data/cleaned/reviews_clean.csv")

# Build combined review text
reviews_text = reviews_clean[
    [
        "order_id",
        "review_id",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    ]
].copy()

reviews_text["review_text"] = (
    reviews_text["review_comment_title"].fillna("").astype(str).str.strip()
    + " "
    + reviews_text["review_comment_message"].fillna("").astype(str).str.strip()
).str.strip()

# Keep reviews with usable text
reviews_text = reviews_text[reviews_text["review_text"].ne("")].copy()

# Keep reviews with at least 3 alphabetic characters
alphabetic_chars = reviews_text["review_text"].str.count(r"[A-Za-zÀ-ÿ]")
nlp_reviews = reviews_text[alphabetic_chars >= 3].copy()

# Create a balanced validation sample by review score
validation_sample = (
    nlp_reviews
    .groupby("review_score", group_keys=False)
    .apply(lambda x: x.sample(
        n=min(50, len(x)),
        random_state=42
    ))
    .reset_index(drop=True)
)

print("Validation sample size:", len(validation_sample))
print("\nReview-score distribution:")
print(validation_sample["review_score"].value_counts().sort_index())

display(
    validation_sample[
        ["review_score", "review_text"]
    ].head(20)
)

Validation sample size: 250

Review-score distribution:
review_score
1    50
2    50
3    50
4    50
5    50
Name: count, dtype: int64


C:\Users\Kamakshi Kunisetty\AppData\Local\Temp\ipykernel_11580\148914990.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


,review_score,review_text
0,1,"O meu produto veio erradi, não foi o que eu pedi, e não estou conseguindo solicitar a troca"
1,1,"Material horrível ,totalmente diferente do que apresenta na imagem e na descrição , não vale o valor ."
2,1,"O pincel chegou quebrado, não fiquei satisfeita com a mercadoria"
3,1,Produto está até hoje na unidade do correio em Cajamar e não chegou em minha casa. Estraviou?
4,1,"Comprei 3 unidades desta cortina, e chegou só 1, não chegaram nem depois, oque também não faz sentido . Sem contar que é de péssima qualidade deveriam avisar que é muito inferior descartável."
5,1,ainda não recebi o produto. ninguém entrou em contato. NÃo recomendo a ninguém . sou cliente baratheon mt tempo primeira vez que acontece essa situação.
6,1,"Produto Horrível A capinha veio completamente danificada e rasurada, com os cantos rasgados e toda defeituosa, a película não encaixou corretamente na tela, ficou menor, os dois produtos horríveis."
7,1,sem comentários.
8,1,Produto que não tem atualização de status e falta de comunicação da empresa com o cliente
9,1,"O PRODUTO CHEGOU ANTES DO PRAZO, POREM COM DEFEITO. LIGA, MÁS NÃO EXERCE SUA FUNÇÕES, ATRIBUO SUPOSTAMENTE O DEFEITO A MÁ QUALIDADE NO EMBALAMENTO, POIS CAIXA CHEGOU BASTANTE AMASSADA."


In [30]:
# Phase 6B — Generate model predictions on validation sample

# Make sure the sentiment pipeline exists
print("Running sentiment model on", len(validation_sample), "reviews...")

validation_results = sentiment_pipeline(
    validation_sample["review_text"].tolist(),
    truncation=True,
    batch_size=16
)

# Store model predictions
validation_sample["model_sentiment"] = [
    result["label"] for result in validation_results
]

validation_sample["model_confidence"] = [
    result["score"] for result in validation_results
]

print("Prediction complete.")

print("\nModel sentiment distribution:")
print(
    validation_sample["model_sentiment"]
    .value_counts()
)

print("\nAverage model confidence:")
print(
    validation_sample["model_confidence"].mean()
)

display(
    validation_sample[
        [
            "review_score",
            "review_text",
            "model_sentiment",
            "model_confidence"
        ]
    ].head(20)
)

Running sentiment model on 250 reviews...
Prediction complete.

Model sentiment distribution:
model_sentiment
NEU    94
POS    81
NEG    75
Name: count, dtype: int64

Average model confidence:
0.8268031463623047


,review_score,review_text,model_sentiment,model_confidence
0,1,"O meu produto veio erradi, não foi o que eu pedi, e não estou conseguindo solicitar a troca",NEG,0.885661
1,1,"Material horrível ,totalmente diferente do que apresenta na imagem e na descrição , não vale o valor .",NEG,0.991591
2,1,"O pincel chegou quebrado, não fiquei satisfeita com a mercadoria",NEG,0.974061
3,1,Produto está até hoje na unidade do correio em Cajamar e não chegou em minha casa. Estraviou?,NEG,0.697192
4,1,"Comprei 3 unidades desta cortina, e chegou só 1, não chegaram nem depois, oque também não faz sentido . Sem contar que é de péssima qualidade deveriam avisar que é muito inferior descartável.",NEG,0.990549
5,1,ainda não recebi o produto. ninguém entrou em contato. NÃo recomendo a ninguém . sou cliente baratheon mt tempo primeira vez que acontece essa situação.,NEG,0.851778
6,1,"Produto Horrível A capinha veio completamente danificada e rasurada, com os cantos rasgados e toda defeituosa, a película não encaixou corretamente na tela, ficou menor, os dois produtos horríveis.",NEG,0.993314
7,1,sem comentários.,NEU,0.561178
8,1,Produto que não tem atualização de status e falta de comunicação da empresa com o cliente,NEG,0.911178
9,1,"O PRODUTO CHEGOU ANTES DO PRAZO, POREM COM DEFEITO. LIGA, MÁS NÃO EXERCE SUA FUNÇÕES, ATRIBUO SUPOSTAMENTE O DEFEITO A MÁ QUALIDADE NO EMBALAMENTO, POIS CAIXA CHEGOU BASTANTE AMASSADA.",NEG,0.872286


In [31]:
# Phase 6B — Compare AI sentiment with rating-based sentiment

def score_to_sentiment(score):
    if score <= 2:
        return "NEG"
    elif score == 3:
        return "NEU"
    else:
        return "POS"

validation_sample["rating_sentiment"] = (
    validation_sample["review_score"]
    .apply(score_to_sentiment)
)

# Agreement
agreement = (
    validation_sample["model_sentiment"]
    == validation_sample["rating_sentiment"]
)

print("Model vs rating-based baseline")
print("=" * 60)

print(f"Agreement: {agreement.sum()} / {len(agreement)}")
print(f"Agreement rate: {agreement.mean():.2%}")

print("\nConfusion matrix:")
confusion = pd.crosstab(
    validation_sample["rating_sentiment"],
    validation_sample["model_sentiment"],
    rownames=["Rating-based sentiment"],
    colnames=["Model sentiment"]
)

display(confusion)

print("\nDisagreement count:")
print((~agreement).sum())

print("\nSample disagreements:")
display(
    validation_sample.loc[
        ~agreement,
        [
            "review_score",
            "rating_sentiment",
            "review_text",
            "model_sentiment",
            "model_confidence"
        ]
    ].head(30)
)

Model vs rating-based baseline
Agreement: 143 / 250
Agreement rate: 57.20%

Confusion matrix:


Model sentiment,NEG,NEU,POS
Rating-based sentiment,,,
NEG,54,44,2
NEU,16,22,12
POS,5,28,67



Disagreement count:
107

Sample disagreements:


,review_score,rating_sentiment,review_text,model_sentiment,model_confidence
7,1,NEG,sem comentários.,NEU,0.561178
11,1,NEG,Faltou item descrito\r\nMe ligue 34 99978 4951,NEU,0.932393
17,1,NEG,a pulseira é muito dificil de abrir!,NEU,0.685901
18,1,NEG,"O produto chegou, mas não foi o comprado. A descrição é a requerida, as fotos do mostruário também, mas o produto entregue foi diferente.",NEU,0.895003
20,1,NEG,Não entregou e não entrou em contato,NEU,0.758928
21,1,NEG,Comprei um recibie outro,NEU,0.910194
24,1,NEG,Meu produto nao foi entregue. Verificar com urgencia.,NEU,0.770867
26,1,NEG,"Até o momento, recebi apenas um dos produtos. Falta entregar o outro.",NEU,0.895442
27,1,NEG,"Fiz 02 (DUAS) solicitações de CANCELAMENTO dessa compra, e ambas foram ignoradas. Mesmo com 02 (DOIS) pedidos de cancelamento o produto foi entregue na portaria do meu prédio.",NEU,0.792327
28,1,NEG,Espera! Produto ainda nao foi entregue. Ainda continuo no aguardo.,NEU,0.920127


In [32]:
# Phase 6B — Analyze sentiment disagreement by review score

agreement_summary = (
    validation_sample
    .assign(
        agreement=
        validation_sample["model_sentiment"]
        == validation_sample["rating_sentiment"]
    )
    .groupby("review_score")
    .agg(
        total_reviews=("review_score", "size"),
        agreements=("agreement", "sum")
    )
    .reset_index()
)

agreement_summary["disagreement_rate"] = (
    1 - agreement_summary["agreements"]
    / agreement_summary["total_reviews"]
)

agreement_summary["agreement_rate"] = (
    agreement_summary["agreements"]
    / agreement_summary["total_reviews"]
)

display(agreement_summary)

print("\nModel sentiment by review score:")

score_model_table = pd.crosstab(
    validation_sample["review_score"],
    validation_sample["model_sentiment"]
)

display(score_model_table)

,review_score,total_reviews,agreements,disagreement_rate,agreement_rate
0,1,50,30,0.40,0.60
1,2,50,24,0.52,0.48
2,3,50,22,0.56,0.44
3,4,50,25,0.50,0.50
4,5,50,42,0.16,0.84



Model sentiment by review score:


model_sentiment,NEG,NEU,POS
review_score,,,
1,30,19,1
2,24,25,1
3,16,22,12
4,5,20,25
5,0,8,42


### AI Sentiment Model Validation

A Portuguese sentiment classification model (`pysentimiento/bertweet-pt-sentiment`) was tested on a balanced validation sample of 250 Olist reviews, containing 50 reviews from each 1–5 star rating.

The model's predictions were compared with a rating-derived sentiment baseline:
- 1–2 stars → Negative
- 3 stars → Neutral
- 4–5 stars → Positive

The model agreed with the rating-derived baseline for 143 of 250 reviews (57.2%).

Agreement varied substantially by rating:
- 1 star: 60%
- 2 stars: 48%
- 3 stars: 44%
- 4 stars: 84%
- 5 stars: 84%

Inspection of disagreements showed that the model frequently classified fulfillment-related complaints as Neutral, including reviews describing missing, incorrect, or undelivered products.

The rating-derived baseline is not treated as ground-truth linguistic sentiment because star ratings and written sentiment measure related but different aspects of customer feedback.

### Decision

The sentiment model will **not be used as the primary source of sentiment labels for the full review dataset**. Instead, review scores will remain the primary quantitative satisfaction measure, while review text will be analyzed for specific customer complaint themes and issues.

This validation demonstrates that AI outputs were evaluated before being used in the business analysis.

# Phase 7 — Customer Complaint Taxonomy

## Objective

Convert unstructured customer review text into a structured set of business-relevant complaint categories.

The taxonomy is designed around recurring issues observed in the Olist review data and the project's customer-experience questions.

## Complaint Categories

| Category | Definition |
|---|---|
| Late Delivery | Customer explicitly reports that the order arrived later than expected or delivery took too long. |
| Product Not Received | Customer reports that the order/product was not received. |
| Missing Product / Item | Customer received the order but one or more expected items were missing. |
| Wrong Product | Customer received a different product, model, size, color, or item than ordered. |
| Product Damage / Defect | Product arrived damaged, broken, defective, or malfunctioning. |
| Product Quality | Complaint about material, durability, appearance, performance, or overall product quality. |
| Seller / Service Issue | Complaint about seller communication, responsiveness, support, or handling of the order. |
| Packaging | Complaint specifically related to packaging, wrapping, or protection of the product. |
| Shipping / Logistics | Complaint about carrier, tracking, postal handling, shipping process, or shipment status that is not specifically a delivery-delay complaint. |
| Payment | Complaint related to payment, charges, installments, refunds, or payment processing. |
| Cancellation / Refund | Complaint involving cancellation, return, exchange, refund, or failure to resolve a cancellation/refund request. |
| Other | Relevant complaint that does not fit the defined categories. |

## Classification Principles

1. A review may contain **multiple complaint categories**.
2. Categories should be assigned based on the **meaning of the review**, not isolated keywords alone.
3. A review should not be classified as a complaint merely because it contains a negative word.
4. Delivery-related categories should be distinguished:
   - **Late Delivery** → received late.
   - **Product Not Received** → not received.
   - **Shipping / Logistics** → tracking/carrier/shipping-process issue.
5. Product-related categories should also be distinguished:
   - **Missing Product / Item** → expected item was absent.
   - **Wrong Product** → incorrect item was received.
   - **Product Damage / Defect** → physical damage or functional defect.
   - **Product Quality** → dissatisfaction with material, durability, appearance, or performance.
6. Multiple categories are allowed when a review describes multiple issues.
7. The taxonomy is a working analytical framework and will be refined if systematic evidence from the review sample shows that categories are too broad, overlapping, or missing an important recurring issue.

## Business Purpose

The taxonomy allows customer complaints to be quantified and connected with structured operational data.

For example:

**Review complaint → Delivery data → Seller → Product category → Geography → Business action**

This transforms unstructured customer feedback into an analyzable source of operational insight.

In [34]:
# Phase 7.1 — Prepare reviews for complaint classification

# Focus on written reviews, with extra emphasis on low-rated experiences
complaint_sample = (
    nlp_reviews[
        nlp_reviews["review_score"] <= 3
    ]
    .sample(
        n=min(300, len(nlp_reviews[nlp_reviews["review_score"] <= 3])),
        random_state=42
    )
    .reset_index(drop=True)
)

print("Complaint classification sample:", len(complaint_sample))
print("\nReview-score distribution:")
print(complaint_sample["review_score"].value_counts().sort_index())

display(
    complaint_sample[
        ["review_score", "review_text"]
    ].head(20)
)

Complaint classification sample: 300

Review-score distribution:
review_score
1    185
2     43
3     72
Name: count, dtype: int64


,review_score,review_text
0,1,"O Produto veio correto, porém com defeito. As lentes menores estão trincadas."
1,2,achei o protudo bom .porem no tem muita potencia.
2,3,Empresa cumpriu com o prazo de entrega e produto bem embalado.
3,1,Já passou o prazo de entrega e nada de produto.
4,1,"Cor entregue errada Comprei uma manta lilás e me enviaram uma cor goiaba,odiei a compra,era um presente e acabei nao dando o presente,e joguei no lixo.Que falta de profissionalismo, pelo jeito o vendedor e bem amador."
5,1,"Compei uma pingente e nunca chegou, comprei dia 06 de janeiro hj é 08 de fervereiro e nada."
6,1,Produto não foi entregue. Não foi enviada nota fiscal. Telefone para reclamação não funciona.
7,1,"NÃO RECOMENDO O PRODUTO VEIO COM DEFEITO E NÃO ME DEVOLVEM O DINHEIRO. CANCELEI O PEDIDO, DEPOIS DO RECEIBIMENTO, DENTRO DOS 2 DIAS UTEIS E ATÉ AGORA NÃO FUI ATENDIDO."
8,3,"Meio satisfeita Fiquei mt satiafeita, mas pensei q era um produto e veio outro. Pq pela foto parecia q era uma colcha e veio no material de lençol. Mas td bem. Nao tem problema. Sempre comprei neste site e confio."
9,3,Recebi outro produto pois oque comprei nao consta em estoque. Mas mesmo assim nao tiraram do site.


In [35]:
%pip install -U google-genai

Note: you may need to restart the kernel to use updated packages.


In [79]:
# Phase 7.2 — Test Gemini API connection

import os
from google import genai

api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError(
        "GEMINI_API_KEY was not found. "
        "Check the Windows environment variable and restart Jupyter."
    )

print("API key detected:", api_key[:6] + "..." + api_key[-4:])

client = genai.Client()

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Reply with exactly: Gemini connection successful."
)

print("\nGemini response:")
print(response.text)

RuntimeError: GEMINI_API_KEY was not found. Check the Windows environment variable and restart Jupyter.

In [80]:
# Phase 7.2 — Secure Gemini connection

import getpass
from google import genai

# Enter your Gemini API key when prompted.
# It will NOT be displayed on screen.
api_key = getpass.getpass("Enter your Gemini API key: ")

if not api_key:
    raise ValueError("No API key entered.")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Reply with exactly: Gemini connection successful."
)

print("Gemini connection successful.")
print("Response:", response.text)

KeyboardInterrupt: Interrupted by user

In [73]:
# Phase 7.2 — Test Gemini with an alternative Flash model

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Reply with exactly: Gemini connection successful."
)

print("Gemini connection successful.")
print("Response:", response.text)

NameError: name 'client' is not defined

In [74]:
# Phase 7.2 — Test Gemini 3.6 Flash

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Reply with exactly: Gemini connection successful."
)

print("Gemini connection successful.")
print("Response:", response.text)

NameError: name 'client' is not defined

In [75]:
# Phase 7.2 — Gemini Complaint Classification Pilot

import json

pilot_reviews = complaint_sample.head(5)[
    ["review_id", "order_id", "review_score", "review_text"]
].copy()

taxonomy = """
1. Late Delivery — product was received after the expected delivery date
2. Product Not Received — product/order was never received
3. Missing Product / Item — one or more expected items were missing
4. Wrong Product — customer received a different product than ordered
5. Product Damage / Defect — product arrived damaged, broken, defective, or unusable
6. Product Quality — product quality/performance did not meet expectations
7. Seller / Service Issue — seller/customer service/responsiveness issue
8. Packaging — packaging was inadequate or damaged
9. Shipping / Logistics — shipping, carrier, tracking, or logistics issue not specifically about lateness
10. Payment — payment, charge, refund-payment, or billing issue
11. Cancellation / Refund — cancellation or refund problem
12. Other — complaint that does not clearly fit the categories above
"""

reviews_for_prompt = []

for _, row in pilot_reviews.iterrows():
    reviews_for_prompt.append({
        "review_id": row["review_id"],
        "order_id": row["order_id"],
        "review_score": int(row["review_score"]),
        "review_text": row["review_text"]
    })

prompt = f"""
You are a customer-experience data analyst classifying Brazilian e-commerce
customer reviews.

Use ONLY the review text provided. Do not invent facts.

Complaint taxonomy:
{taxonomy}

Important classification rules:
- A review can contain multiple complaint categories.
- Do not assign a complaint category merely because the review has a low star rating.
- Some low-rated reviews may contain no clear complaint.
- If there is no clear complaint, set is_complaint to false and categories to [].
- Distinguish "Late Delivery" from "Product Not Received".
- Distinguish "Missing Product / Item" from "Wrong Product".
- Distinguish "Product Damage / Defect" from general "Product Quality".
- Severity:
    High = major failure such as non-delivery, serious defect, or major fulfilment problem
    Medium = meaningful problem affecting the customer experience
    Low = minor issue or dissatisfaction
    None = no clear complaint
- complaint_summary must be concise and based only on the review.

Return ONLY valid JSON in this exact structure:

[
  {{
    "review_id": "...",
    "order_id": "...",
    "is_complaint": true,
    "categories": ["Late Delivery"],
    "severity": "Medium",
    "complaint_summary": "Short factual summary"
  }}
]

Reviews:
{json.dumps(reviews_for_prompt, ensure_ascii=False, indent=2)}
"""

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt,
    config={
        "response_mime_type": "application/json"
    }
)

pilot_results = json.loads(response.text)

pilot_results

NameError: name 'client' is not defined

In [76]:
# Recreate the NLP review data and 300-review complaint sample

import pandas as pd

reviews_clean = pd.read_csv("../data/cleaned/reviews_clean.csv")

reviews_text = reviews_clean[
    ["order_id", "review_id", "review_score",
     "review_comment_title", "review_comment_message"]
].copy()

reviews_text["review_text"] = (
    reviews_text["review_comment_title"].fillna("").astype(str).str.strip()
    + " "
    + reviews_text["review_comment_message"].fillna("").astype(str).str.strip()
).str.strip()

reviews_text["has_review_text"] = reviews_text["review_text"].ne("")

# Keep reviews with meaningful alphabetic text
nlp_reviews = reviews_text[
    reviews_text["review_text"].str.count(r"[A-Za-zÀ-ÿ]") >= 3
].copy()

# 300 low-rated written reviews for AI classification
complaint_sample = (
    nlp_reviews[nlp_reviews["review_score"] <= 3]
    .sample(n=300, random_state=42)
    .reset_index(drop=True)
)

print("NLP-ready reviews:", len(nlp_reviews))
print("Complaint validation sample:", len(complaint_sample))
print("\nRating distribution:")
print(complaint_sample["review_score"].value_counts().sort_index())

NLP-ready reviews: 42288
Complaint validation sample: 300

Rating distribution:
review_score
1    185
2     43
3     72
Name: count, dtype: int64


In [40]:
# Test Gemini 3.7 Flash for the complaint-classification pilot

response = client.models.generate_content(
    model="gemini-3.7-flash",
    contents="Reply with exactly: Complaint classification model ready."
)

print(response.text)

NameError: name 'client' is not defined

In [41]:
# Check models available to this API client

available_models = []

for model in client.models.list():
    if "generateContent" in (model.supported_actions or []):
        available_models.append(model.name)

print("Models available for generateContent:")
for model in available_models:
    print(model)

NameError: name 'client' is not defined

In [42]:
# Phase 7.2 — Gemini 3.5 Flash functional test

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Reply with exactly: Complaint classification model ready."
)

print(response.text)

NameError: name 'client' is not defined

In [43]:
# Phase 7.2 — Transparent Rule-Based Complaint Screening Baseline

import re
import pandas as pd

# Keyword patterns based on the finalized complaint taxonomy
complaint_patterns = {
    "Late Delivery": [
        r"\batras", r"\batraso", r"\batrasada", r"\batrasado",
        r"\bdemora", r"\bdemorou", r"\btarde", r"\bprazo.*entrega",
        r"\bentrega.*prazo", r"\bchegou.*tarde", r"\bchegou.*atras"
    ],

    "Product Not Received": [
        r"\bn[aã]o.*cheg", r"\bnunca.*cheg", r"\bn[aã]o.*receb",
        r"\bn[aã]o.*entreg", r"\bproduto.*n[aã]o.*cheg",
        r"\bpedido.*n[aã]o.*cheg", r"\bpedido.*n[aã]o.*receb"
    ],

    "Missing Product / Item": [
        r"\bfaltou", r"\bfaltando", r"\bfaltaram", r"\bitens?.*falt",
        r"\bproduto.*falt", r"\bveio.*falt", r"\bn[aã]o.*veio"
    ],

    "Wrong Product": [
        r"\bproduto.*errad", r"\bveio.*errad", r"\bpedido.*errad",
        r"\breceb.*errad", r"\bdiferente.*an[uú]ncio",
        r"\bn[aã]o.*foi.*comprad"
    ],

    "Product Damage / Defect": [
        r"\bdefeit", r"\bquebrad", r"\bdanific", r"\bdanificad",
        r"\bdanificado", r"\bavari", r"\bn[aã]o.*funcion",
        r"\bparou.*funcion", r"\bdefeituos"
    ],

    "Product Quality": [
        r"\bqualidade", r"\bqualidad.*ruim", r"\bfr[aá]gil",
        r"\bpot[eê]ncia", r"\bmaterial.*ruim", r"\bacabamento",
        r"\bpior.*esper", r"\bbaixa.*qualidade"
    ],

    "Seller / Service Issue": [
        r"\bvendedor", r"\bloja", r"\batendimento", r"\bsuporte",
        r"\bcontato", r"\bresposta", r"\bresponder", r"\bempresa.*n[aã]o",
        r"\bsem.*resposta"
    ],

    "Packaging": [
        r"\bembalagem", r"\bembalado", r"\bcaixa.*dan",
        r"\bcaixa.*queb", r"\bmal.*embalad"
    ],

    "Shipping / Logistics": [
        r"\btransportadora", r"\btransport", r"\brastreio",
        r"\brastreamento", r"\bcorreio", r"\bfrete",
        r"\blog[ií]stic"
    ],

    "Payment": [
        r"\bpagamento", r"\bcobran[cç]a", r"\bcart[aã]o",
        r"\bparcel", r"\bpag[ouo]", r"\bvalor.*cobr"
    ],

    "Cancellation / Refund": [
        r"\bcancel", r"\breembolso", r"\bdevolu[cç]",
        r"\bdinheiro.*volta", r"\bestorno"
    ]
}


def classify_keywords(text):
    text = str(text).lower()
    
    matched_categories = []
    
    for category, patterns in complaint_patterns.items():
        if any(re.search(pattern, text) for pattern in patterns):
            matched_categories.append(category)
    
    return matched_categories


# Apply to the 300-review validation sample
complaint_sample["keyword_categories"] = (
    complaint_sample["review_text"]
    .apply(classify_keywords)
)

complaint_sample["keyword_is_complaint"] = (
    complaint_sample["keyword_categories"].str.len() > 0
)

# Summary
print("Total reviews:", len(complaint_sample))
print(
    "Reviews flagged as potential complaints:",
    complaint_sample["keyword_is_complaint"].sum()
)
print(
    "Reviews with no keyword match:",
    (~complaint_sample["keyword_is_complaint"]).sum()
)

print("\nCategory frequency:")
category_counts = (
    complaint_sample
    .explode("keyword_categories")["keyword_categories"]
    .dropna()
    .value_counts()
)

print(category_counts)

Total reviews: 300
Reviews flagged as potential complaints: 217
Reviews with no keyword match: 83

Category frequency:
keyword_categories
Product Not Received       102
Seller / Service Issue      51
Late Delivery               32
Product Damage / Defect     27
Cancellation / Refund       25
Missing Product / Item      23
Shipping / Logistics        22
Payment                     22
Product Quality             18
Wrong Product               11
Packaging                   10
Name: count, dtype: int64


In [44]:
# Phase 9 — Validate "Product Not Received" complaints against order data

import pandas as pd

# Load cleaned orders if not already available
orders_clean = pd.read_csv("../data/cleaned/orders_clean.csv")

# Get the reviews flagged as Product Not Received
not_received = complaint_sample[
    complaint_sample["keyword_categories"].apply(
        lambda x: "Product Not Received" in x
    )
].copy()

# Keep only the fields needed for validation
validation = not_received[
    ["review_id", "order_id", "review_score", "review_text"]
].merge(
    orders_clean[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ],
    on="order_id",
    how="left"
)

print("Product Not Received review sample:", len(validation))

print("\nOrder status distribution:")
print(validation["order_status"].value_counts(dropna=False))

print("\nDelivery date availability:")
print(
    validation[
        ["order_delivered_customer_date",
         "order_estimated_delivery_date"]
    ].notna().sum()
)

print("\nSample records:")
display(
    validation[
        [
            "review_score",
            "order_status",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "review_text"
        ]
    ].head(15)
)

Product Not Received review sample: 102

Order status distribution:
order_status
delivered      81
canceled        8
unavailable     5
shipped         4
processing      2
invoiced        2
Name: count, dtype: int64

Delivery date availability:
order_delivered_customer_date     82
order_estimated_delivery_date    102
dtype: int64

Sample records:


,review_score,order_status,order_delivered_customer_date,order_estimated_delivery_date,review_text
0,1,delivered,2018-02-20 22:42:53,2018-02-05,"Compei uma pingente e nunca chegou, comprei dia 06 de janeiro hj é 08 de fervereiro e nada."
1,1,canceled,2016-10-19 18:47:43,2016-11-30,Produto não foi entregue. Não foi enviada nota fiscal. Telefone para reclamação não funciona.
2,1,delivered,2018-04-10 23:58:57,2018-05-04,Bom dia. Comprei 2 produtos 01 ventilador de teto que ainda não recebi. e um filtro do squime que fui buscar no correio mesmo pagando frete.
3,1,delivered,2018-09-27 02:24:33,2018-08-17,Produto não entregue Pensei que seria entregue antes do prazo mas nenhuma coisa nenhuma outra
4,1,canceled,NaN,2018-03-21,Nao recebi o produto.
5,1,delivered,2017-01-30 18:37:44,2017-02-28,MINHA COMPRA NAO CHEGOU!!
6,1,delivered,2017-02-22 11:03:15,2017-03-20,Mandei email não respondi. Produto disse que está entregue. Mas estou pedindo assinatura pra mim ver se alguem pode ter recebido em meu lugar email não é respondido
7,3,delivered,2018-01-13 02:18:48,2018-01-19,"A loja vendeu e entregou tudo certo, porem não compro mais, o correio nao entrega em casa. Fui buscar na central, as duas vezes teve barraco, e paguei pra entrega ksa, nao me sujeito mais protesto."
8,1,delivered,2018-05-03 15:46:14,2018-04-27,Atraso grande O vendedor não entregou o produto. Pedi à lannister que fizesse as devidas cobranças.
9,3,delivered,2018-01-11 20:19:37,2018-01-26,"não sei se eu não prestei a atenção devida quanto ao tipo do papel na hora da compra: eu esperava receber um papel de 230g, e veio um de 180g, mais a entrega foi em tempo abil. Obrigado e bom dia."


In [45]:
# Phase 9 — Refine "Product Not Received" hypothesis using delivery performance

delivered_validation = validation[
    validation["order_status"] == "delivered"
].copy()

# Convert dates
delivered_validation["delivered_date"] = pd.to_datetime(
    delivered_validation["order_delivered_customer_date"],
    errors="coerce"
)

delivered_validation["estimated_date"] = pd.to_datetime(
    delivered_validation["order_estimated_delivery_date"],
    errors="coerce"
)

# Calculate delivery delay
delivered_validation["delivery_delay_days"] = (
    delivered_validation["delivered_date"]
    - delivered_validation["estimated_date"]
).dt.total_seconds() / (60 * 60 * 24)

delivered_validation["delivery_result"] = delivered_validation[
    "delivery_delay_days"
].apply(
    lambda x: (
        "Late"
        if x > 0
        else "On Time / Early"
    )
    if pd.notna(x)
    else "Unknown"
)

print("Delivered reviews:", len(delivered_validation))

print("\nDelivery result:")
print(delivered_validation["delivery_result"].value_counts())

print("\nAverage delivery delay:")
print(
    delivered_validation["delivery_delay_days"]
    .describe()
)

print("\nDelivery result percentages:")
print(
    delivered_validation["delivery_result"]
    .value_counts(normalize=True).mul(100).round(2)
)

Delivered reviews: 81

Delivery result:
delivery_result
Late               47
On Time / Early    34
Name: count, dtype: int64

Average delivery delay:
count    81.000000
mean      3.529948
std      20.247021
min     -32.202188
25%     -11.973113
50%       4.678252
75%      13.725613
max      91.753565
Name: delivery_delay_days, dtype: float64

Delivery result percentages:
delivery_result
Late               58.02
On Time / Early    41.98
Name: proportion, dtype: float64


In [46]:
# Phase 9 — Validate Seller / Service complaints against seller performance

seller_complaints = complaint_sample[
    complaint_sample["keyword_categories"].apply(
        lambda x: "Seller / Service Issue" in x
    )
].copy()

print("Seller / Service complaint sample:", len(seller_complaints))

# Load required cleaned tables
order_items_clean = pd.read_csv("../data/cleaned/order_items_clean.csv")
sellers_clean = pd.read_csv("../data/cleaned/sellers_clean.csv")
reviews_clean = pd.read_csv("../data/cleaned/reviews_clean.csv")

# Connect review → order → seller
seller_validation = (
    seller_complaints[
        ["review_id", "order_id", "review_score", "review_text"]
    ]
    .merge(
        order_items_clean[
            ["order_id", "seller_id", "price", "freight_value"]
        ],
        on="order_id",
        how="left"
    )
)

# Seller-level performance
seller_performance = (
    order_items_clean
    .groupby("seller_id")
    .agg(
        total_orders=("order_id", "nunique"),
        total_items=("order_item_id", "count"),
        total_revenue=("price", "sum"),
        avg_freight=("freight_value", "mean")
    )
    .reset_index()
)

# Add seller performance to complaint records
seller_validation = seller_validation.merge(
    seller_performance,
    on="seller_id",
    how="left"
)

print("\nUnique sellers represented:", seller_validation["seller_id"].nunique())

print("\nComplaint records by seller:")
print(
    seller_validation["seller_id"]
    .value_counts()
    .head(15)
)

print("\nSeller performance summary:")
display(
    seller_validation[
        [
            "seller_id",
            "total_orders",
            "total_revenue",
            "avg_freight"
        ]
    ]
    .drop_duplicates("seller_id")
    .sort_values("total_orders", ascending=False)
    .head(15)
)

Seller / Service complaint sample: 51

Unique sellers represented: 49

Complaint records by seller:
seller_id
f45122a9ab94eb4f3f8953578bc0c560    3
1900267e848ceeba8fa32d80c1a5f5a8    2
b2ba3715d723d245138f291a6fe42594    2
0ea22c1cfbdc755f86b9b54b39c16043    2
1e8b33f18b4f7598d87f5cbee2282cc2    2
2138ccb85b11a4ec1e37afbd1c8eda1f    2
ef506c96320abeedfb894c34db06f478    2
1a3df491d1c4f1589fc2b934ada68bf2    2
602044f2c16190c2c6e45eb35c2e21cb    2
634964b17796e64304cadf1ad3050fb7    2
c4fb51fb1c5b7c07bc5e67be6e7e8f6e    1
f8db351d8c4c4c22c6835c19a46f01b0    1
ea8482cd71df3c1969d7b9473ff13abc    1
014d9a685fd57276679edd00e07089e5    1
0725b8c0f3f906e58f70cbe76b7c748c    1
Name: count, dtype: int64

Seller performance summary:


,seller_id,total_orders,total_revenue,avg_freight
59,6560211a19b47992c3666cc44a7e94c0,1854.0,123304.83,13.753537
9,4a3ca9315b744ce9f8e9374361493884,1806.0,200472.92,17.648234
35,955fee9216a65b617aa5c0531780ce60,1287.0,135171.70,16.965297
36,7a67c85e85bb2ce8582c35f2203ad736,1160.0,141745.53,17.850427
39,ea8482cd71df3c1969d7b9473ff13abc,1146.0,37177.52,14.584181
47,4869f7a5dfa277a7dca6462dcf3b52b2,1132.0,229472.63,17.446427
38,f8db351d8c4c4c22c6835c19a46f01b0,667.0,50525.60,17.340055
22,fa1c13f2614d7b5c4749cbc52fecda94,585.0,194042.03,17.137713
0,1900267e848ceeba8fa32d80c1a5f5a8,411.0,24982.53,15.383856
7,dbc22125167c298ef99da25668e1011f,406.0,33776.78,18.676876


In [47]:
# Phase 9 — Seller / Service complaints vs overall seller satisfaction
# Corrected: derive seller_id through order_items

# Connect complaint reviews → orders → sellers
seller_complaints = (
    complaint_sample[
        ["review_id", "order_id", "review_score", "review_text"]
    ]
    .merge(
        order_items_clean[
            ["order_id", "seller_id"]
        ],
        on="order_id",
        how="left"
    )
    .drop_duplicates(
        subset=["review_id", "order_id", "seller_id"]
    )
)

# Keep only reviews flagged as Seller / Service Issue
seller_complaints = seller_complaints[
    complaint_sample.set_index("review_id")
    .loc[
        seller_complaints["review_id"],
        "keyword_categories"
    ]
    .apply(lambda x: "Seller / Service Issue" in x)
    .values
].copy()

print("Seller/service complaint records linked to sellers:",
      len(seller_complaints))

print("Unique sellers represented:",
      seller_complaints["seller_id"].nunique())


# Overall seller customer-review performance
seller_review_performance = (
    order_items_clean[
        ["order_id", "seller_id"]
    ]
    .merge(
        reviews_clean[
            ["order_id", "review_score"]
        ],
        on="order_id",
        how="inner"
    )
    .groupby("seller_id")
    .agg(
        reviewed_orders=("order_id", "nunique"),
        avg_review_score=("review_score", "mean"),
        low_rating_rate=(
            "review_score",
            lambda x: (x <= 2).mean() * 100
        )
    )
    .reset_index()
)

# Compare complaint-associated sellers with their overall performance
complaint_sellers = (
    seller_complaints[["seller_id"]]
    .drop_duplicates()
    .merge(
        seller_review_performance,
        on="seller_id",
        how="left"
    )
)

print("\nOverall performance of complaint-associated sellers:")

display(
    complaint_sellers[
        [
            "seller_id",
            "reviewed_orders",
            "avg_review_score",
            "low_rating_rate"
        ]
    ]
    .sort_values("avg_review_score")
    .head(20)
)

print("\nSummary:")
display(
    complaint_sellers[
        [
            "reviewed_orders",
            "avg_review_score",
            "low_rating_rate"
        ]
    ].describe()
)

Seller/service complaint records linked to sellers: 55
Unique sellers represented: 49

Overall performance of complaint-associated sellers:


,seller_id,reviewed_orders,avg_review_score,low_rating_rate
22,b1b3948701c5c72445495bd161b83a4c,18.0,1.722222,77.777778
4,602044f2c16190c2c6e45eb35c2e21cb,49.0,2.932203,45.762712
30,a49928bcdf77c55c6d6e05e09a9b4ca5,98.0,2.952830,43.396226
17,082e0bf4cb865a6533b1e8e498cc0255,11.0,3.304348,30.434783
20,2ea0861cc19e94cad86438c984c52da4,6.0,3.333333,16.666667
13,88460e8ebdecbfecb5f9601833981930,247.0,3.351613,32.258065
38,c4fb51fb1c5b7c07bc5e67be6e7e8f6e,15.0,3.500000,18.750000
34,0725b8c0f3f906e58f70cbe76b7c748c,17.0,3.575000,32.500000
15,ef990a83bbea832f36ebe81376335aa8,43.0,3.659091,25.000000
14,25c5c91f63607446a97b143d2d535d31,157.0,3.703297,22.710623



Summary:


,reviewed_orders,avg_review_score,low_rating_rate
count,49.000000,49.000000,49.000000
mean,321.673469,3.897371,19.152117
std,441.136193,0.481847,11.959586
min,6.000000,1.722222,4.545455
25%,67.000000,3.741935,11.990950
50%,162.000000,3.909406,16.666667
75%,334.000000,4.226107,21.276596
max,1838.000000,4.606061,77.777778


In [48]:
# Phase 9 — Validate Cancellation / Refund complaints against order and payment data

cancellation_complaints = complaint_sample[
    complaint_sample["keyword_categories"].apply(
        lambda x: "Cancellation / Refund" in x
    )
].copy()

print("Cancellation / Refund complaint reviews:",
      len(cancellation_complaints))

# Connect reviews to order information
cancellation_validation = (
    cancellation_complaints[
        ["review_id", "order_id", "review_score", "review_text"]
    ]
    .merge(
        orders_clean[
            [
                "order_id",
                "order_status",
                "order_purchase_timestamp",
                "order_delivered_customer_date"
            ]
        ],
        on="order_id",
        how="left"
    )
    .merge(
        payments_clean[
            [
                "order_id",
                "payment_value",
                "payment_type",
                "payment_installments"
            ]
        ],
        on="order_id",
        how="left"
    )
)

print("\nOrder status distribution:")
print(
    cancellation_validation["order_status"]
    .value_counts(dropna=False)
)

print("\nPayment information availability:")
print(
    cancellation_validation[
        ["payment_value", "payment_type"]
    ].notna().sum()
)

print("\nSample:")
display(
    cancellation_validation[
        [
            "review_score",
            "order_status",
            "payment_value",
            "payment_type",
            "review_text"
        ]
    ].head(20)
)

Cancellation / Refund complaint reviews: 25


NameError: name 'payments_clean' is not defined

In [49]:
# ============================================================
# PHASE 8 — HYPOTHESIS GENERATION
# ============================================================

import pandas as pd

hypotheses = pd.DataFrame([
    {
        "Hypothesis_ID": "H1",
        "Hypothesis": "Late delivery is strongly associated with lower customer satisfaction.",
        "Evidence_Source": "Delivery delay vs review score",
        "Current_Evidence": "Kruskal-Wallis p < 0.001; epsilon-squared = 0.0901",
        "Status": "Strong candidate for validation"
    },
    {
        "Hypothesis_ID": "H2",
        "Hypothesis": "Greater delivery distance is associated with lower customer satisfaction.",
        "Evidence_Source": "Estimated geographic distance vs review score",
        "Current_Evidence": "Kruskal-Wallis p < 0.001; epsilon-squared = 0.00438",
        "Status": "Candidate; effect is small"
    },
    {
        "Hypothesis_ID": "H3",
        "Hypothesis": "Higher order value is associated with lower customer satisfaction.",
        "Evidence_Source": "Order value vs review score",
        "Current_Evidence": "Kruskal-Wallis p < 0.001; epsilon-squared = 0.00176",
        "Status": "Candidate; effect is very small"
    },
    {
        "Hypothesis_ID": "H4",
        "Hypothesis": "Higher freight burden is associated with lower customer satisfaction.",
        "Evidence_Source": "Freight ratio vs review score",
        "Current_Evidence": "Kruskal-Wallis p < 0.001; epsilon-squared = 0.00066",
        "Status": "Candidate; effect is extremely small"
    },
    {
        "Hypothesis_ID": "H5",
        "Hypothesis": "Specific sellers, product categories, or geographic areas may contain concentrated customer-experience problems.",
        "Evidence_Source": "Seller, category, geographic and review-theme analyses",
        "Current_Evidence": "Requires structured-data validation and business-level comparison",
        "Status": "Candidate for Phase 9"
    }
])

display(hypotheses)

,Hypothesis_ID,Hypothesis,Evidence_Source,Current_Evidence,Status
0,H1,Late delivery is strongly associated with lower customer satisfaction.,Delivery delay vs review score,Kruskal-Wallis p < 0.001; epsilon-squared = 0.0901,Strong candidate for validation
1,H2,Greater delivery distance is associated with lower customer satisfaction.,Estimated geographic distance vs review score,Kruskal-Wallis p < 0.001; epsilon-squared = 0.00438,Candidate; effect is small
2,H3,Higher order value is associated with lower customer satisfaction.,Order value vs review score,Kruskal-Wallis p < 0.001; epsilon-squared = 0.00176,Candidate; effect is very small
3,H4,Higher freight burden is associated with lower customer satisfaction.,Freight ratio vs review score,Kruskal-Wallis p < 0.001; epsilon-squared = 0.00066,Candidate; effect is extremely small
4,H5,"Specific sellers, product categories, or geographic areas may contain concentrated customer-experience problems.","Seller, category, geographic and review-theme analyses",Requires structured-data validation and business-level comparison,Candidate for Phase 9


In [50]:
# ============================================================
# PHASE 9 — AI VS DATA
# ============================================================

phase9_evidence = pd.DataFrame([
    {
        "Hypothesis_ID": "H1",
        "Hypothesis": "Late delivery is strongly associated with lower customer satisfaction.",
        "Data_Test": "Compare review-score distributions across delivery-delay severity bands.",
        "Statistical_Test": "Kruskal-Wallis + pairwise Mann-Whitney U with Holm correction",
        "Sample_Size": 95824,
        "P_Value": "< 0.001",
        "Effect_Size_Epsilon_Squared": 0.09011,
        "Evidence_Interpretation": "Statistically significant association with a measurable effect.",
        "Conclusion": "Supported as an association; not evidence of causation."
    },
    {
        "Hypothesis_ID": "H2",
        "Hypothesis": "Greater delivery distance is associated with lower customer satisfaction.",
        "Data_Test": "Compare review-score distributions across geographic-distance bands.",
        "Statistical_Test": "Kruskal-Wallis + pairwise Mann-Whitney U with Holm correction",
        "Sample_Size": 97430,
        "P_Value": "< 0.001",
        "Effect_Size_Epsilon_Squared": 0.00438,
        "Evidence_Interpretation": "Statistically significant but small association.",
        "Conclusion": "Supported as a weak association; distance is not a major standalone explanation."
    },
    {
        "Hypothesis_ID": "H3",
        "Hypothesis": "Higher order value is associated with lower customer satisfaction.",
        "Data_Test": "Compare review-score distributions across order-value bands.",
        "Statistical_Test": "Kruskal-Wallis + pairwise Mann-Whitney U with Holm correction",
        "Sample_Size": 97917,
        "P_Value": "< 0.001",
        "Effect_Size_Epsilon_Squared": 0.00176,
        "Evidence_Interpretation": "Statistically significant but very small association.",
        "Conclusion": "Supported as a weak association; order value is not a strong standalone explanation."
    },
    {
        "Hypothesis_ID": "H4",
        "Hypothesis": "Higher freight burden is associated with lower customer satisfaction.",
        "Data_Test": "Compare review-score distributions across freight-ratio bands.",
        "Statistical_Test": "Kruskal-Wallis + pairwise Mann-Whitney U with Holm correction",
        "Sample_Size": 97917,
        "P_Value": "< 0.001",
        "Effect_Size_Epsilon_Squared": 0.00066,
        "Evidence_Interpretation": "Statistically significant but extremely small association.",
        "Conclusion": "Supported as a weak association; freight burden is not a major standalone explanation."
    },
    {
        "Hypothesis_ID": "H5",
        "Hypothesis": "Specific sellers, product categories, or geographic areas may contain concentrated customer-experience problems.",
        "Data_Test": "Compare seller, category and geographic customer-experience metrics.",
        "Statistical_Test": "Descriptive and comparative analysis; further validation where appropriate.",
        "Sample_Size": None,
        "P_Value": None,
        "Effect_Size_Epsilon_Squared": None,
        "Evidence_Interpretation": "Requires consolidation of seller, category and geographic findings.",
        "Conclusion": "To be evaluated in the dedicated Seller, Category and Geographic analysis phases."
    }
])

display(phase9_evidence)

,Hypothesis_ID,Hypothesis,Data_Test,Statistical_Test,Sample_Size,P_Value,Effect_Size_Epsilon_Squared,Evidence_Interpretation,Conclusion
0,H1,Late delivery is strongly associated with lower customer satisfaction.,Compare review-score distributions across delivery-delay severity bands.,Kruskal-Wallis + pairwise Mann-Whitney U with Holm correction,95824.0,< 0.001,0.09011,Statistically significant association with a measurable effect.,Supported as an association; not evidence of causation.
1,H2,Greater delivery distance is associated with lower customer satisfaction.,Compare review-score distributions across geographic-distance bands.,Kruskal-Wallis + pairwise Mann-Whitney U with Holm correction,97430.0,< 0.001,0.00438,Statistically significant but small association.,Supported as a weak association; distance is not a major standalone explanation.
2,H3,Higher order value is associated with lower customer satisfaction.,Compare review-score distributions across order-value bands.,Kruskal-Wallis + pairwise Mann-Whitney U with Holm correction,97917.0,< 0.001,0.00176,Statistically significant but very small association.,Supported as a weak association; order value is not a strong standalone explanation.
3,H4,Higher freight burden is associated with lower customer satisfaction.,Compare review-score distributions across freight-ratio bands.,Kruskal-Wallis + pairwise Mann-Whitney U with Holm correction,97917.0,< 0.001,0.00066,Statistically significant but extremely small association.,Supported as a weak association; freight burden is not a major standalone explanation.
4,H5,"Specific sellers, product categories, or geographic areas may contain concentrated customer-experience problems.","Compare seller, category and geographic customer-experience metrics.",Descriptive and comparative analysis; further validation where appropriate.,NaN,None,NaN,"Requires consolidation of seller, category and geographic findings.","To be evaluated in the dedicated Seller, Category and Geographic analysis phases."


In [51]:
# ============================================================
# PHASE 11 — SELLER PERFORMANCE & OPERATIONAL ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

# Load cleaned / analytical data
order_items_clean = pd.read_csv("../data/cleaned/order_items_clean.csv")
orders_clean = pd.read_csv("../data/cleaned/orders_clean.csv")
reviews_clean = pd.read_csv("../data/cleaned/reviews_clean.csv")

# ------------------------------------------------------------
# 1. Seller order / revenue / freight metrics
# ------------------------------------------------------------

seller_sales = (
    order_items_clean
    .groupby("seller_id")
    .agg(
        total_orders=("order_id", "nunique"),
        total_items=("order_item_id", "count"),
        total_revenue=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
    .reset_index()
)

seller_sales["freight_ratio"] = (
    seller_sales["total_freight"] /
    seller_sales["total_revenue"]
)

# ------------------------------------------------------------
# 2. Seller delivery performance
# ------------------------------------------------------------

orders_delivery = orders_clean[
    [
        "order_id",
        "order_status",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].copy()

orders_delivery["order_delivered_customer_date"] = pd.to_datetime(
    orders_delivery["order_delivered_customer_date"],
    errors="coerce"
)

orders_delivery["order_estimated_delivery_date"] = pd.to_datetime(
    orders_delivery["order_estimated_delivery_date"],
    errors="coerce"
)

orders_delivery["delivery_delay_days"] = (
    orders_delivery["order_delivered_customer_date"]
    - orders_delivery["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

orders_delivery["is_late"] = (
    orders_delivery["delivery_delay_days"] > 0
)

# Connect orders to sellers.
# An order may contain multiple sellers, so seller-order combinations
# are treated as the unit for seller operational analysis.
seller_orders = (
    order_items_clean[
        ["order_id", "seller_id"]
    ]
    .drop_duplicates()
    .merge(
        orders_delivery,
        on="order_id",
        how="left"
    )
)

# Only orders with a known delivery outcome are used for delivery metrics
seller_delivery = (
    seller_orders[
        seller_orders["delivery_delay_days"].notna()
    ]
    .groupby("seller_id")
    .agg(
        delivery_orders=("order_id", "nunique"),
        late_orders=("is_late", "sum"),
        avg_delivery_delay_days=("delivery_delay_days", "mean")
    )
    .reset_index()
)

seller_delivery["late_delivery_rate"] = (
    seller_delivery["late_orders"] /
    seller_delivery["delivery_orders"] * 100
)

# ------------------------------------------------------------
# 3. Seller customer satisfaction
# ------------------------------------------------------------

seller_reviews = (
    order_items_clean[
        ["order_id", "seller_id"]
    ]
    .drop_duplicates()
    .merge(
        reviews_clean[
            ["order_id", "review_score"]
        ],
        on="order_id",
        how="inner"
    )
)

seller_satisfaction = (
    seller_reviews
    .groupby("seller_id")
    .agg(
        reviewed_orders=("order_id", "nunique"),
        avg_review_score=("review_score", "mean"),
        negative_reviews=(
            "review_score",
            lambda x: (x <= 2).sum()
        )
    )
    .reset_index()
)

seller_satisfaction["negative_review_rate"] = (
    seller_satisfaction["negative_reviews"] /
    seller_satisfaction["reviewed_orders"] * 100
)

# ------------------------------------------------------------
# 4. Build final seller analytical dataset
# ------------------------------------------------------------

seller_performance = (
    seller_sales
    .merge(
        seller_delivery,
        on="seller_id",
        how="left"
    )
    .merge(
        seller_satisfaction,
        on="seller_id",
        how="left"
    )
)

# ------------------------------------------------------------
# 5. Quality checks
# ------------------------------------------------------------

print("Seller analytical dataset shape:",
      seller_performance.shape)

print(
    "Unique sellers:",
    seller_performance["seller_id"].nunique()
)

print(
    "\nDuplicate seller IDs:",
    seller_performance["seller_id"].duplicated().sum()
)

print("\nMissing values:")
print(
    seller_performance.isna().sum()
)

print("\nSeller performance preview:")
display(
    seller_performance.head(15)
)

Seller analytical dataset shape: (3095, 14)
Unique sellers: 3095

Duplicate seller IDs: 0

Missing values:
seller_id                    0
total_orders                 0
total_items                  0
total_revenue                0
total_freight                0
freight_ratio                0
delivery_orders            125
late_orders                125
avg_delivery_delay_days    125
late_delivery_rate         125
reviewed_orders              5
avg_review_score             5
negative_reviews             5
negative_review_rate         5
dtype: int64

Seller performance preview:


,seller_id,total_orders,total_items,total_revenue,total_freight,freight_ratio,delivery_orders,late_orders,avg_delivery_delay_days,late_delivery_rate,reviewed_orders,avg_review_score,negative_reviews,negative_review_rate
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3,2685.00,63.06,0.023486,3.0,0.0,-15.593380,0.000000,3.0,3.666667,1.0,33.333333
1,001cca7ae9ae17fb1caed9dfb1094831,200,239,25080.03,8854.14,0.353035,195.0,13.0,-12.225614,6.666667,197.0,3.984772,32.0,16.243655
2,001e6ad469a905060d959994f1b41e4f,1,1,250.00,17.94,0.071760,NaN,NaN,NaN,NaN,1.0,1.000000,1.0,100.000000
3,002100f778ceb8431b7a1020ff7ab48f,51,55,1234.50,793.66,0.642900,50.0,9.0,-7.389838,18.000000,51.0,3.903846,8.0,15.686275
4,003554e2dce176b5555353e4f3555ac8,1,1,120.00,19.38,0.161500,1.0,0.0,-26.066794,0.000000,1.0,5.000000,0.0,0.000000
5,004c9cd9d87a3c30c522c48c4fc07416,158,170,19712.71,3551.23,0.180149,156.0,13.0,-11.063157,8.333333,155.0,4.136646,22.0,14.193548
6,00720abe85ba0859807595bbf045a33b,13,26,1007.50,315.98,0.313628,13.0,2.0,-7.076197,15.384615,13.0,3.615385,3.0,23.076923
7,00ab3eff1b5192e5f1a63bcecfee11c8,1,1,98.00,12.08,0.123265,1.0,0.0,-9.461100,0.000000,1.0,5.000000,0.0,0.000000
8,00d8b143d12632bad99c0ad66ad52825,1,1,86.00,51.10,0.594186,1.0,0.0,-21.296088,0.000000,1.0,5.000000,0.0,0.000000
9,00ee68308b45bc5e2660cd833c3f81cc,135,172,20260.00,3180.66,0.156992,135.0,12.0,-10.066531,8.888889,134.0,4.303704,15.0,11.194030


In [52]:
# ============================================================
# PHASE 11 — SELLER PERFORMANCE SEGMENTATION
# ============================================================

# Minimum reviewed orders for meaningful satisfaction comparison
MIN_REVIEWS = 30

seller_analysis = seller_performance.copy()

# Flag sellers with enough review volume for comparison
seller_analysis["reliable_review_volume"] = (
    seller_analysis["reviewed_orders"] >= MIN_REVIEWS
)

# Revenue and customer-experience benchmarks
revenue_median = seller_analysis["total_revenue"].median()

# Calculate experience benchmark only among sellers
# with sufficient review volume
experience_median = seller_analysis.loc[
    seller_analysis["reliable_review_volume"],
    "avg_review_score"
].median()

# Segment sellers with reliable review volume
def assign_segment(row):
    if not row["reliable_review_volume"]:
        return "Insufficient Review Volume"
    
    high_revenue = row["total_revenue"] >= revenue_median
    high_experience = row["avg_review_score"] >= experience_median
    
    if high_revenue and high_experience:
        return "High Revenue + High Experience"
    elif high_revenue and not high_experience:
        return "High Revenue + Lower Experience"
    elif not high_revenue and high_experience:
        return "Lower Revenue + High Experience"
    else:
        return "Lower Revenue + Lower Experience"


seller_analysis["seller_segment"] = (
    seller_analysis.apply(assign_segment, axis=1)
)

print("Revenue median:", round(revenue_median, 2))
print("Experience median:", round(experience_median, 3))

print("\nSeller segment distribution:")
print(
    seller_analysis["seller_segment"]
    .value_counts()
)

print("\nHigh Revenue + Lower Experience sellers:")
display(
    seller_analysis[
        seller_analysis["seller_segment"]
        == "High Revenue + Lower Experience"
    ][
        [
            "seller_id",
            "total_orders",
            "total_revenue",
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate",
            "avg_delivery_delay_days",
            "freight_ratio",
            "reviewed_orders"
        ]
    ]
    .sort_values(
        ["total_revenue", "avg_review_score"],
        ascending=[False, True]
    )
    .head(20)
)

Revenue median: 821.48
Experience median: 4.154

Seller segment distribution:
seller_segment
Insufficient Review Volume         2467
High Revenue + Lower Experience     314
High Revenue + High Experience      312
Lower Revenue + High Experience       2
Name: count, dtype: int64

High Revenue + Lower Experience sellers:


,seller_id,total_orders,total_revenue,avg_review_score,negative_review_rate,late_delivery_rate,avg_delivery_delay_days,freight_ratio,reviewed_orders
857,4869f7a5dfa277a7dca6462dcf3b52b2,1132,229472.63,4.133452,13.523132,11.565836,-10.390535,0.087889,1124.0
1013,53243585a1d6dc2643021fd1853d8905,358,222776.05,4.132022,12.078652,4.310345,-10.743274,0.058717,356.0
881,4a3ca9315b744ce9f8e9374361493884,1806,200472.92,3.827873,19.327731,11.004515,-8.831082,0.174922,1785.0
1535,7c67e1448b00f6e969d365cea6b010ab,982,187923.89,3.488798,25.922131,10.071942,-10.691665,0.274646,976.0
192,1025f0e2d44d7041d6cf58b6550e0bfa,915,138968.55,3.989059,17.640573,10.109890,-10.182349,0.243884,907.0
1235,6560211a19b47992c3666cc44a7e94c0,1854,123304.83,3.937093,17.355822,6.432106,-10.791104,0.226763,1838.0
1540,7d13fca15225358621be4086e1eb0964,565,113628.97,4.024955,17.468806,12.186380,-10.172528,0.075967,561.0
1153,5dceca129747e92ff8ef7a997dc4f8ca,325,112155.53,3.984424,19.937695,6.211180,-10.619047,0.121808,321.0
368,1f50f920176fa81dab994f9023523100,1404,106939.21,4.130342,14.295926,10.578985,-9.634086,0.328839,1399.0
2481,cc419e0650a3c5ba77189a1882b7556a,1706,104288.42,4.077586,15.253239,6.117505,-12.288448,0.246135,1698.0


In [53]:
# ============================================================
# PHASE 11 — SELLER SEGMENT SUMMARY
# ============================================================

segment_summary = (
    seller_analysis
    .groupby("seller_segment")
    .agg(
        sellers=("seller_id", "nunique"),
        total_orders=("total_orders", "sum"),
        total_revenue=("total_revenue", "sum"),
        avg_review_score=("avg_review_score", "mean"),
        avg_negative_review_rate=("negative_review_rate", "mean"),
        avg_late_delivery_rate=("late_delivery_rate", "mean")
    )
    .reset_index()
)

# Revenue contribution
total_revenue = seller_analysis["total_revenue"].sum()

segment_summary["revenue_share_pct"] = (
    segment_summary["total_revenue"] /
    total_revenue * 100
)

print("SELLER SEGMENT SUMMARY")
display(
    segment_summary.sort_values(
        "total_revenue",
        ascending=False
    )
)

# Save the reusable seller dataset
seller_analysis.to_csv(
    "../data/analytical/seller_performance.csv",
    index=False
)

print(
    "\nSaved: ../data/analytical/seller_performance.csv"
)

SELLER SEGMENT SUMMARY


,seller_segment,sellers,total_orders,total_revenue,avg_review_score,avg_negative_review_rate,avg_late_delivery_rate,revenue_share_pct
1,High Revenue + Lower Experience,314,47370,5465284.46,3.882989,19.340995,9.908010,40.210622
0,High Revenue + High Experience,312,35703,4942197.65,4.347027,8.941276,5.642653,36.362031
2,Insufficient Review Volume,2467,16873,3183114.45,3.968649,18.720952,8.597874,23.419643
3,Lower Revenue + High Experience,2,64,1047.14,4.838710,0.000000,1.612903,0.007704



Saved: ../data/analytical/seller_performance.csv


In [78]:
# Save complaint taxonomy results for Power BI

complaint_analysis = complaint_sample[
    [
        "review_id",
        "order_id",
        "review_score",
        "review_text",
        "keyword_is_complaint",
        "keyword_categories"
    ]
].copy()

# One row per review-category combination
complaint_analysis = complaint_analysis.explode(
    "keyword_categories"
).reset_index(drop=True)

# Label reviews with no detected complaint
complaint_analysis["complaint_category"] = (
    complaint_analysis["keyword_categories"]
    .fillna("No Clear Complaint")
)

# Remove temporary column
complaint_analysis = complaint_analysis.drop(
    columns=["keyword_categories"]
)

# Save for Power BI
complaint_analysis.to_csv(
    "../data/analytical/complaint_analysis.csv",
    index=False
)

print("Complaint analysis saved successfully.")
print("Rows:", len(complaint_analysis))
print("Unique reviews:", complaint_analysis["review_id"].nunique())

print("\nCategory counts:")
print(
    complaint_analysis["complaint_category"]
    .value_counts()
)

KeyError: "['keyword_is_complaint', 'keyword_categories'] not in index"

In [54]:
# ============================================================
# PHASE 12 — PRODUCT CATEGORY ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load cleaned data
# ------------------------------------------------------------

order_items_clean = pd.read_csv(
    "../data/cleaned/order_items_clean.csv"
)

orders_clean = pd.read_csv(
    "../data/cleaned/orders_clean.csv"
)

reviews_clean = pd.read_csv(
    "../data/cleaned/reviews_clean.csv"
)

products_clean = pd.read_csv(
    "../data/cleaned/products_clean.csv"
)

# Correct filename
category_translation = pd.read_csv(
    "../data/cleaned/category_translation_clean.csv"
)

# ------------------------------------------------------------
# 2. Attach product category to order items
# ------------------------------------------------------------

category_items = (
    order_items_clean
    .merge(
        products_clean[
            ["product_id", "product_category_name"]
        ],
        on="product_id",
        how="left"
    )
    .merge(
        category_translation,
        on="product_category_name",
        how="left"
    )
)

# Use English category when available.
# Preserve original Portuguese category when translation
# is unavailable.
category_items["category"] = (
    category_items["product_category_name_english"]
    .fillna(category_items["product_category_name"])
)

# ------------------------------------------------------------
# 3. Category sales metrics
# ------------------------------------------------------------

category_sales = (
    category_items
    .groupby("category", dropna=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_items=("order_item_id", "count"),
        total_revenue=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
    .reset_index()
)

category_sales["freight_ratio"] = (
    category_sales["total_freight"] /
    category_sales["total_revenue"]
)

# ------------------------------------------------------------
# 4. Category delivery performance
# ------------------------------------------------------------

category_orders = (
    category_items[
        ["order_id", "category"]
    ]
    .drop_duplicates()
    .merge(
        orders_clean[
            [
                "order_id",
                "order_delivered_customer_date",
                "order_estimated_delivery_date"
            ]
        ],
        on="order_id",
        how="left"
    )
)

category_orders[
    "order_delivered_customer_date"
] = pd.to_datetime(
    category_orders["order_delivered_customer_date"],
    errors="coerce"
)

category_orders[
    "order_estimated_delivery_date"
] = pd.to_datetime(
    category_orders["order_estimated_delivery_date"],
    errors="coerce"
)

category_orders["delivery_delay_days"] = (
    category_orders["order_delivered_customer_date"]
    - category_orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

category_delivery = (
    category_orders[
        category_orders["delivery_delay_days"].notna()
    ]
    .groupby("category")
    .agg(
        delivery_orders=("order_id", "nunique"),
        late_orders=(
            "delivery_delay_days",
            lambda x: (x > 0).sum()
        ),
        avg_delivery_delay_days=(
            "delivery_delay_days",
            "mean"
        )
    )
    .reset_index()
)

category_delivery["late_delivery_rate"] = (
    category_delivery["late_orders"] /
    category_delivery["delivery_orders"] * 100
)

# ------------------------------------------------------------
# 5. Category customer satisfaction
# ------------------------------------------------------------

category_reviews = (
    category_items[
        ["order_id", "category"]
    ]
    .drop_duplicates()
    .merge(
        reviews_clean[
            ["order_id", "review_score"]
        ],
        on="order_id",
        how="inner"
    )
)

category_satisfaction = (
    category_reviews
    .groupby("category")
    .agg(
        reviewed_orders=("order_id", "nunique"),
        avg_review_score=("review_score", "mean"),
        negative_reviews=(
            "review_score",
            lambda x: (x <= 2).sum()
        )
    )
    .reset_index()
)

category_satisfaction["negative_review_rate"] = (
    category_satisfaction["negative_reviews"] /
    category_satisfaction["reviewed_orders"] * 100
)

# ------------------------------------------------------------
# 6. Build final category analytical dataset
# ------------------------------------------------------------

category_performance = (
    category_sales
    .merge(
        category_delivery,
        on="category",
        how="left"
    )
    .merge(
        category_satisfaction,
        on="category",
        how="left"
    )
)

# ------------------------------------------------------------
# 7. Quality checks
# ------------------------------------------------------------

print(
    "Category analytical dataset shape:",
    category_performance.shape
)

print(
    "Unique categories:",
    category_performance["category"].nunique()
)

print(
    "\nDuplicate categories:",
    category_performance["category"].duplicated().sum()
)

print("\nMissing values:")
print(category_performance.isna().sum())

print("\nCategory performance preview:")
display(
    category_performance
    .sort_values(
        "total_revenue",
        ascending=False
    )
    .head(20)
)

Category analytical dataset shape: (74, 14)
Unique categories: 73

Duplicate categories: 0

Missing values:
category                   1
total_orders               0
total_items                0
total_revenue              0
total_freight              0
freight_ratio              0
delivery_orders            1
late_orders                1
avg_delivery_delay_days    1
late_delivery_rate         1
reviewed_orders            1
avg_review_score           1
negative_reviews           1
negative_review_rate       1
dtype: int64

Category performance preview:


,category,total_orders,total_items,total_revenue,total_freight,freight_ratio,delivery_orders,late_orders,avg_delivery_delay_days,late_delivery_rate,reviewed_orders,avg_review_score,negative_reviews,negative_review_rate
43,health_beauty,8836,9670,1258681.34,182566.73,0.145046,8649.0,776.0,-11.281708,8.972136,8771.0,4.182189,1118.0,12.746551
72,watches_gifts,5624,5991,1205005.68,100535.93,0.083432,5493.0,468.0,-11.092277,8.519934,5576.0,4.065700,849.0,15.225968
7,bed_bath_table,9417,11115,1036988.68,204693.04,0.197392,9272.0,811.0,-10.693906,8.746764,9313.0,3.968617,1587.0,17.040696
67,sports_leisure,7720,8641,988048.97,168607.51,0.170647,7530.0,584.0,-11.168421,7.755644,7669.0,4.168242,1003.0,13.078628
15,computers_accessories,6689,7827,911954.32,147318.08,0.161541,6529.0,503.0,-11.682958,7.704089,6649.0,4.025813,1078.0,16.212964
39,furniture_decor,6449,8334,729762.49,172749.30,0.236720,6307.0,535.0,-11.654866,8.482638,6398.0,4.006040,1081.0,16.895905
20,cool_stuff,3632,3796,635290.85,84039.10,0.132284,3559.0,243.0,-11.732877,6.827761,3599.0,4.169299,452.0,12.559044
49,housewares,5884,6964,632248.66,146149.11,0.231158,5743.0,399.0,-11.395985,6.947588,5843.0,4.142370,773.0,13.229505
5,auto,3897,4235,592720.11,92664.21,0.156337,3809.0,328.0,-10.509993,8.611184,3877.0,4.090629,564.0,14.547330
42,garden_tools,3518,4347,485256.46,98962.75,0.203939,3448.0,274.0,-11.164895,7.946636,3496.0,4.134341,473.0,13.529748


In [55]:
# Correct category translation file name

category_translation = pd.read_csv(
    "../data/cleaned/category_translation_clean.csv"
)

print("Category translation shape:", category_translation.shape)
display(category_translation.head())

Category translation shape: (71, 2)


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [56]:
# ============================================================
# PHASE 12 — CATEGORY PERFORMANCE ANALYSIS
# ============================================================

category_analysis = category_performance.copy()

# Preserve missing-category records explicitly
category_analysis["category"] = (
    category_analysis["category"]
    .fillna("Unclassified / Missing Category")
)

# ------------------------------------------------------------
# 1. Identify categories with sufficient review volume
# ------------------------------------------------------------

MIN_REVIEWS = 100

category_analysis["reliable_review_volume"] = (
    category_analysis["reviewed_orders"] >= MIN_REVIEWS
)

# Only real named categories for benchmark comparison
benchmark_categories = category_analysis[
    (category_analysis["reliable_review_volume"]) &
    (category_analysis["category"] != "Unclassified / Missing Category")
].copy()

# ------------------------------------------------------------
# 2. Category benchmarks
# ------------------------------------------------------------

revenue_median = benchmark_categories["total_revenue"].median()
experience_median = benchmark_categories["avg_review_score"].median()

print("Minimum reviews for comparison:", MIN_REVIEWS)
print("Category revenue median:", round(revenue_median, 2))
print("Category average-rating median:", round(experience_median, 3))

# ------------------------------------------------------------
# 3. Revenue + customer-experience segmentation
# ------------------------------------------------------------

def assign_category_segment(row):

    if row["category"] == "Unclassified / Missing Category":
        return "Unclassified / Missing Category"

    if not row["reliable_review_volume"]:
        return "Insufficient Review Volume"

    high_revenue = row["total_revenue"] >= revenue_median
    high_experience = row["avg_review_score"] >= experience_median

    if high_revenue and high_experience:
        return "High Revenue + High Experience"

    elif high_revenue and not high_experience:
        return "High Revenue + Lower Experience"

    elif not high_revenue and high_experience:
        return "Lower Revenue + High Experience"

    else:
        return "Lower Revenue + Lower Experience"


category_analysis["category_segment"] = (
    category_analysis.apply(
        assign_category_segment,
        axis=1
    )
)

# ------------------------------------------------------------
# 4. Segment distribution
# ------------------------------------------------------------

print("\nCategory segment distribution:")
print(
    category_analysis["category_segment"]
    .value_counts()
)

# ------------------------------------------------------------
# 5. High-revenue + lower-experience categories
# ------------------------------------------------------------

high_revenue_lower_experience = (
    category_analysis[
        category_analysis["category_segment"]
        == "High Revenue + Lower Experience"
    ]
    [
        [
            "category",
            "total_orders",
            "total_items",
            "total_revenue",
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate",
            "avg_delivery_delay_days",
            "freight_ratio",
            "reviewed_orders"
        ]
    ]
    .sort_values(
        "total_revenue",
        ascending=False
    )
)

print("\nHigh Revenue + Lower Experience categories:")
display(
    high_revenue_lower_experience
)

# ------------------------------------------------------------
# 6. Overall category performance
# ------------------------------------------------------------

print("\nCategories by negative-review rate:")
display(
    category_analysis[
        category_analysis["reliable_review_volume"]
    ]
    [
        [
            "category",
            "reviewed_orders",
            "total_revenue",
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate",
            "freight_ratio"
        ]
    ]
    .sort_values(
        "negative_review_rate",
        ascending=False
    )
    .head(15)
)

# ------------------------------------------------------------
# 7. Save reusable category dataset
# ------------------------------------------------------------

category_analysis.to_csv(
    "../data/analytical/category_performance.csv",
    index=False
)

print(
    "\nSaved: ../data/analytical/category_performance.csv"
)

Minimum reviews for comparison: 100
Category revenue median: 113317.74
Category average-rating median: 4.109

Category segment distribution:
category_segment
Insufficient Review Volume          22
High Revenue + High Experience      15
Lower Revenue + Lower Experience    14
High Revenue + Lower Experience     11
Lower Revenue + High Experience     11
Unclassified / Missing Category      1
Name: count, dtype: int64

High Revenue + Lower Experience categories:


,category,total_orders,total_items,total_revenue,avg_review_score,negative_review_rate,late_delivery_rate,avg_delivery_delay_days,freight_ratio,reviewed_orders
72,watches_gifts,5624,5991,1205005.68,4.065700,15.225968,8.519934,-11.092277,0.083432,5576.0
7,bed_bath_table,9417,11115,1036988.68,3.968617,17.040696,8.746764,-10.693906,0.197392,9313.0
15,computers_accessories,6689,7827,911954.32,4.025813,16.212964,7.704089,-11.682958,0.161541,6649.0
39,furniture_decor,6449,8334,729762.49,4.006040,16.895905,8.482638,-11.654866,0.236720,6398.0
5,auto,3897,4235,592720.11,4.090629,14.547330,8.611184,-10.509993,0.156337,3877.0
6,baby,2885,3065,411764.89,4.039735,16.008389,9.184763,-10.794724,0.166000,2861.0
70,telephony,4199,4545,323667.53,4.004074,15.475048,8.526753,-10.561554,0.220028,4168.0
57,office_furniture,1273,1691,273960.70,3.617508,22.802850,9.170654,-11.134130,0.250298,1263.0
26,electronics,2550,2767,160246.74,4.095577,14.065587,9.813270,-10.309546,0.290666,2531.0
16,consoles_games,1062,1137,157465.22,4.064577,14.258555,8.153242,-10.828964,0.125920,1052.0



Categories by negative-review rate:


,category,reviewed_orders,total_revenue,avg_review_score,negative_review_rate,late_delivery_rate,freight_ratio
30,fashion_male_clothing,111.0,10797.82,3.702703,26.126126,4.716981,0.199337
57,office_furniture,1263.0,273960.70,3.617508,22.802850,9.170654,0.250298
4,audio,347.0,50688.50,3.827586,22.190202,12.931034,0.112658
19,construction_tools_safety,166.0,40544.52,3.849398,20.481928,4.402516,0.096662
47,home_confort,395.0,58572.04,3.859296,19.493671,10.459184,0.145141
34,fixed_telephony,214.0,59583.00,3.902326,18.691589,5.188679,0.077838
33,fashion_underwear_beach,120.0,9541.55,3.933333,18.333333,12.820513,0.200826
48,home_construction,487.0,83088.12,3.969262,17.659138,8.074534,0.166477
7,bed_bath_table,9313.0,1036988.68,3.968617,17.040696,8.746764,0.197392
39,furniture_decor,6398.0,729762.49,4.006040,16.895905,8.482638,0.236720



Saved: ../data/analytical/category_performance.csv


In [57]:
# ============================================================
# PHASE 12 — FINAL CATEGORY TARGETS
# ============================================================

print("Category segment distribution:")
print(
    category_analysis["category_segment"]
    .value_counts()
)

print("\nHigh Revenue + Lower Experience categories:")

display(
    category_analysis[
        category_analysis["category_segment"]
        == "High Revenue + Lower Experience"
    ][
        [
            "category",
            "total_orders",
            "total_revenue",
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate",
            "avg_delivery_delay_days",
            "freight_ratio",
            "reviewed_orders"
        ]
    ]
    .sort_values(
        "total_revenue",
        ascending=False
    )
)

Category segment distribution:
category_segment
Insufficient Review Volume          22
High Revenue + High Experience      15
Lower Revenue + Lower Experience    14
High Revenue + Lower Experience     11
Lower Revenue + High Experience     11
Unclassified / Missing Category      1
Name: count, dtype: int64

High Revenue + Lower Experience categories:


,category,total_orders,total_revenue,avg_review_score,negative_review_rate,late_delivery_rate,avg_delivery_delay_days,freight_ratio,reviewed_orders
72,watches_gifts,5624,1205005.68,4.065700,15.225968,8.519934,-11.092277,0.083432,5576.0
7,bed_bath_table,9417,1036988.68,3.968617,17.040696,8.746764,-10.693906,0.197392,9313.0
15,computers_accessories,6689,911954.32,4.025813,16.212964,7.704089,-11.682958,0.161541,6649.0
39,furniture_decor,6449,729762.49,4.006040,16.895905,8.482638,-11.654866,0.236720,6398.0
5,auto,3897,592720.11,4.090629,14.547330,8.611184,-10.509993,0.156337,3877.0
6,baby,2885,411764.89,4.039735,16.008389,9.184763,-10.794724,0.166000,2861.0
70,telephony,4199,323667.53,4.004074,15.475048,8.526753,-10.561554,0.220028,4168.0
57,office_furniture,1273,273960.70,3.617508,22.802850,9.170654,-11.134130,0.250298,1263.0
26,electronics,2550,160246.74,4.095577,14.065587,9.813270,-10.309546,0.290666,2531.0
16,consoles_games,1062,157465.22,4.064577,14.258555,8.153242,-10.828964,0.125920,1052.0


In [58]:
# ============================================================
# PHASE 13 — GEOGRAPHIC ANALYSIS
# Cell 1: Build State, City & Region Analytical Datasets
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load cleaned datasets
# ------------------------------------------------------------

orders = pd.read_csv("../data/cleaned/orders_clean.csv")
customers = pd.read_csv("../data/cleaned/customers_clean.csv")
order_items = pd.read_csv("../data/cleaned/order_items_clean.csv")
reviews = pd.read_csv("../data/cleaned/reviews_clean.csv")

# Convert required dates
orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"], errors="coerce"
)
orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"], errors="coerce"
)

# ------------------------------------------------------------
# 2. Create order-level revenue and freight metrics
# ------------------------------------------------------------

order_financials = (
    order_items
    .groupby("order_id")
    .agg(
        total_product_price=("price", "sum"),
        total_freight_value=("freight_value", "sum")
    )
    .reset_index()
)

order_financials["freight_ratio"] = np.where(
    order_financials["total_product_price"] > 0,
    order_financials["total_freight_value"]
    / order_financials["total_product_price"],
    np.nan
)

# ------------------------------------------------------------
# 3. Create order-level review metrics
# ------------------------------------------------------------

order_reviews = (
    reviews
    .groupby("order_id")
    .agg(
        avg_review_score=("review_score", "mean"),
        review_count=("review_score", "count")
    )
    .reset_index()
)

order_reviews["negative_review"] = np.where(
    order_reviews["avg_review_score"] <= 2,
    1,
    0
)

# ------------------------------------------------------------
# 4. Build geographic order-level dataset
# ------------------------------------------------------------

geo_orders = (
    orders[
        [
            "order_id",
            "customer_id",
            "order_status",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ]
    .merge(customers, on="customer_id", how="left")
    .merge(order_financials, on="order_id", how="left")
    .merge(order_reviews, on="order_id", how="left")
)

# ------------------------------------------------------------
# 5. Delivery metrics
# ------------------------------------------------------------

geo_orders["delivery_delay_days"] = (
    geo_orders["order_delivered_customer_date"]
    - geo_orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

# Only orders with a usable actual delivery date can be
# evaluated for delivery performance.
geo_orders["is_delivered"] = (
    geo_orders["order_delivered_customer_date"].notna()
)

geo_orders["is_late"] = pd.Series(
    np.where(
        geo_orders["delivery_delay_days"].isna(),
        pd.NA,
        geo_orders["delivery_delay_days"] > 0
    ),
    dtype="boolean"
)

# ------------------------------------------------------------
# 6. Standardize geographic fields
# ------------------------------------------------------------

geo_orders["customer_state"] = (
    geo_orders["customer_state"]
    .astype("string")
    .str.strip()
    .str.upper()
)

geo_orders["customer_city"] = (
    geo_orders["customer_city"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Missing geographic values are explicitly labelled
geo_orders["customer_state"] = (
    geo_orders["customer_state"]
    .fillna("Unknown")
)

geo_orders["customer_city"] = (
    geo_orders["customer_city"]
    .fillna("Unknown")
)

# ------------------------------------------------------------
# 7. Map Brazilian states to macro-regions
# ------------------------------------------------------------

region_map = {
    # North
    "AC": "North",
    "AP": "North",
    "AM": "North",
    "PA": "North",
    "RO": "North",
    "RR": "North",
    "TO": "North",

    # Northeast
    "AL": "Northeast",
    "BA": "Northeast",
    "CE": "Northeast",
    "MA": "Northeast",
    "PB": "Northeast",
    "PE": "Northeast",
    "PI": "Northeast",
    "RN": "Northeast",
    "SE": "Northeast",

    # Central-West
    "DF": "Central-West",
    "GO": "Central-West",
    "MT": "Central-West",
    "MS": "Central-West",

    # Southeast
    "ES": "Southeast",
    "MG": "Southeast",
    "RJ": "Southeast",
    "SP": "Southeast",

    # South
    "PR": "South",
    "RS": "South",
    "SC": "South"
}

geo_orders["customer_region"] = (
    geo_orders["customer_state"]
    .map(region_map)
    .fillna("Unknown")
)

# ------------------------------------------------------------
# 8. Reusable aggregation function
# ------------------------------------------------------------

def build_geo_performance(df, geo_column):

    result = (
        df.groupby(geo_column, dropna=False)
        .agg(
            total_orders=("order_id", "nunique"),
            total_revenue=("total_product_price", "sum"),
            total_freight=("total_freight_value", "sum"),

            delivery_orders=("is_delivered", "sum"),

            late_orders=("is_late", lambda x: x.eq(True).sum()),

            avg_delivery_delay_days=("delivery_delay_days", "mean"),

            reviewed_orders=("avg_review_score", "count"),

            avg_review_score=("avg_review_score", "mean"),

            negative_reviews=("negative_review", "sum")
        )
        .reset_index()
    )

    # Late-delivery rate
    result["late_delivery_rate"] = np.where(
        result["delivery_orders"] > 0,
        result["late_orders"] / result["delivery_orders"] * 100,
        np.nan
    )

    # Negative-review rate
    result["negative_review_rate"] = np.where(
        result["reviewed_orders"] > 0,
        result["negative_reviews"] / result["reviewed_orders"] * 100,
        np.nan
    )

    # Freight ratio at geographic level
    result["freight_ratio"] = np.where(
        result["total_revenue"] > 0,
        result["total_freight"] / result["total_revenue"],
        np.nan
    )

    return result


# ------------------------------------------------------------
# 9. Build STATE dataset
# ------------------------------------------------------------

state_performance = build_geo_performance(
    geo_orders,
    "customer_state"
)

state_performance = state_performance.rename(
    columns={"customer_state": "state"}
)

# ------------------------------------------------------------
# 10. Build CITY dataset
# ------------------------------------------------------------

city_performance = build_geo_performance(
    geo_orders,
    "customer_city"
)

city_performance = city_performance.rename(
    columns={"customer_city": "city"}
)

# ------------------------------------------------------------
# 11. Build REGION dataset
# ------------------------------------------------------------

region_performance = build_geo_performance(
    geo_orders,
    "customer_region"
)

region_performance = region_performance.rename(
    columns={"customer_region": "region"}
)

# ------------------------------------------------------------
# 12. Save reusable analytical datasets
# ------------------------------------------------------------

state_performance.to_csv(
    "../data/analytical/geographic_state_performance.csv",
    index=False
)

city_performance.to_csv(
    "../data/analytical/geographic_city_performance.csv",
    index=False
)

region_performance.to_csv(
    "../data/analytical/geographic_region_performance.csv",
    index=False
)

# ------------------------------------------------------------
# 13. Basic output
# ------------------------------------------------------------

print("PHASE 13 — GEOGRAPHIC DATASETS CREATED")
print("-" * 55)

print("Order-level geographic dataset:", geo_orders.shape)
print("State dataset:", state_performance.shape)
print("City dataset:", city_performance.shape)
print("Region dataset:", region_performance.shape)

print("\nState columns:")
print(state_performance.columns.tolist())

print("\nRegion summary:")
display(
    region_performance.sort_values(
        "total_orders",
        ascending=False
    )
)

print("\nFiles saved:")
print("✓ geographic_state_performance.csv")
print("✓ geographic_city_performance.csv")
print("✓ geographic_region_performance.csv")

PHASE 13 — GEOGRAPHIC DATASETS CREATED
-------------------------------------------------------
Order-level geographic dataset: (99441, 20)
State dataset: (27, 13)
City dataset: (4119, 13)
Region dataset: (5, 13)

State columns:
['state', 'total_orders', 'total_revenue', 'total_freight', 'delivery_orders', 'late_orders', 'avg_delivery_delay_days', 'reviewed_orders', 'avg_review_score', 'negative_reviews', 'late_delivery_rate', 'negative_review_rate', 'freight_ratio']

Region summary:


,region,total_orders,total_revenue,total_freight,delivery_orders,late_orders,avg_delivery_delay_days,reviewed_orders,avg_review_score,negative_reviews,late_delivery_rate,negative_review_rate,freight_ratio
4,Southeast,68266,8887393.06,1344930.44,66198,4933,-10.860119,67719,4.107941,9671.0,7.451887,14.281073,0.151330
3,South,14148,1953941.12,343034.68,13814,974,-12.381944,14071,4.135207,1876.0,7.050818,13.332386,0.175560
2,Northeast,9394,1545493.75,335305.79,9044,1296,-10.648149,9302,3.895936,1748.0,14.329943,18.791658,0.216957
0,Central-West,5782,870462.06,152599.94,5624,448,-11.636596,5748,4.069415,839.0,7.965861,14.596381,0.175309
1,North,1851,334353.71,76038.69,1796,176,-14.926715,1833,3.956901,315.0,9.799555,17.184943,0.227420



Files saved:
✓ geographic_state_performance.csv
✓ geographic_city_performance.csv
✓ geographic_region_performance.csv


In [59]:
# ============================================================
# PHASE 13 — GEOGRAPHIC ANALYSIS
# Cell 2: Geographic Hotspot Analysis
# ============================================================

# ------------------------------------------------------------
# 1. Load the reusable geographic datasets
# ------------------------------------------------------------

state_performance = pd.read_csv(
    "../data/analytical/geographic_state_performance.csv"
)

city_performance = pd.read_csv(
    "../data/analytical/geographic_city_performance.csv"
)

region_performance = pd.read_csv(
    "../data/analytical/geographic_region_performance.csv"
)

# ------------------------------------------------------------
# 2. State-level overview
# ------------------------------------------------------------

print("STATE PERFORMANCE — CUSTOMER EXPERIENCE")
print("-" * 70)

state_overview = (
    state_performance
    .sort_values("total_orders", ascending=False)
    [
        [
            "state",
            "total_orders",
            "total_revenue",
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate",
            "freight_ratio"
        ]
    ]
)

display(state_overview)

# ------------------------------------------------------------
# 3. Define a meaningful state volume threshold
# ------------------------------------------------------------
# We avoid calling a very small state a hotspot based on
# unstable percentages.

STATE_MIN_ORDERS = 500

states_for_hotspot = state_performance[
    state_performance["total_orders"] >= STATE_MIN_ORDERS
].copy()

# Dataset-wide benchmarks
state_avg_review = state_performance["avg_review_score"].mean()
state_avg_negative = state_performance["negative_review_rate"].mean()
state_avg_late = state_performance["late_delivery_rate"].mean()
state_avg_freight = state_performance["freight_ratio"].mean()

# ------------------------------------------------------------
# 4. Identify state-level hotspot candidates
# ------------------------------------------------------------

states_for_hotspot["poor_review_signal"] = (
    states_for_hotspot["avg_review_score"] < state_avg_review
)

states_for_hotspot["negative_review_signal"] = (
    states_for_hotspot["negative_review_rate"] > state_avg_negative
)

states_for_hotspot["late_delivery_signal"] = (
    states_for_hotspot["late_delivery_rate"] > state_avg_late
)

states_for_hotspot["freight_signal"] = (
    states_for_hotspot["freight_ratio"] > state_avg_freight
)

states_for_hotspot["hotspot_signals"] = (
    states_for_hotspot[
        [
            "poor_review_signal",
            "negative_review_signal",
            "late_delivery_signal",
            "freight_signal"
        ]
    ]
    .sum(axis=1)
)

state_hotspots = (
    states_for_hotspot
    .sort_values(
        ["hotspot_signals", "total_orders"],
        ascending=[False, False]
    )
)

print("\nSTATE HOTSPOT CANDIDATES")
print("-" * 70)

display(
    state_hotspots[
        [
            "state",
            "total_orders",
            "total_revenue",
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate",
            "freight_ratio",
            "hotspot_signals"
        ]
    ]
)

# ------------------------------------------------------------
# 5. City-level analysis
# ------------------------------------------------------------
# Use a minimum order threshold so tiny cities do not dominate
# the analysis through unstable rates.

CITY_MIN_ORDERS = 100

cities_for_hotspot = city_performance[
    (city_performance["total_orders"] >= CITY_MIN_ORDERS) &
    (city_performance["city"] != "unknown")
].copy()

# City-level benchmarks
city_avg_review = city_performance["avg_review_score"].mean()
city_avg_negative = city_performance["negative_review_rate"].mean()
city_avg_late = city_performance["late_delivery_rate"].mean()
city_avg_freight = city_performance["freight_ratio"].mean()

cities_for_hotspot["poor_review_signal"] = (
    cities_for_hotspot["avg_review_score"] < city_avg_review
)

cities_for_hotspot["negative_review_signal"] = (
    cities_for_hotspot["negative_review_rate"] > city_avg_negative
)

cities_for_hotspot["late_delivery_signal"] = (
    cities_for_hotspot["late_delivery_rate"] > city_avg_late
)

cities_for_hotspot["freight_signal"] = (
    cities_for_hotspot["freight_ratio"] > city_avg_freight
)

cities_for_hotspot["hotspot_signals"] = (
    cities_for_hotspot[
        [
            "poor_review_signal",
            "negative_review_signal",
            "late_delivery_signal",
            "freight_signal"
        ]
    ]
    .sum(axis=1)
)

city_hotspots = (
    cities_for_hotspot
    .sort_values(
        ["hotspot_signals", "total_orders"],
        ascending=[False, False]
    )
)

print("\nCITY HOTSPOT CANDIDATES")
print("-" * 70)

display(
    city_hotspots.head(25)[
        [
            "city",
            "total_orders",
            "total_revenue",
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate",
            "freight_ratio",
            "hotspot_signals"
        ]
    ]
)

# ------------------------------------------------------------
# 6. High-volume + poor-delivery cities
# ------------------------------------------------------------

high_volume_poor_delivery = (
    cities_for_hotspot[
        cities_for_hotspot["late_delivery_rate"] > city_avg_late
    ]
    .sort_values("total_orders", ascending=False)
)

print("\nHIGH-VOLUME CITIES WITH ABOVE-AVERAGE LATE DELIVERY")
print("-" * 70)

display(
    high_volume_poor_delivery.head(15)[
        [
            "city",
            "total_orders",
            "total_revenue",
            "late_delivery_rate",
            "avg_review_score",
            "negative_review_rate"
        ]
    ]
)

# ------------------------------------------------------------
# 7. High-volume + poor customer satisfaction cities
# ------------------------------------------------------------

high_volume_poor_cx = (
    cities_for_hotspot[
        (
            cities_for_hotspot["avg_review_score"] < city_avg_review
        )
        &
        (
            cities_for_hotspot["negative_review_rate"] > city_avg_negative
        )
    ]
    .sort_values("total_orders", ascending=False)
)

print("\nHIGH-VOLUME CITIES WITH WEAKER CUSTOMER EXPERIENCE")
print("-" * 70)

display(
    high_volume_poor_cx.head(15)[
        [
            "city",
            "total_orders",
            "total_revenue",
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate",
            "freight_ratio"
        ]
    ]
)

# ------------------------------------------------------------
# 8. Save hotspot datasets for future analysis
# ------------------------------------------------------------

state_hotspots.to_csv(
    "../data/analytical/geographic_state_hotspots.csv",
    index=False
)

city_hotspots.to_csv(
    "../data/analytical/geographic_city_hotspots.csv",
    index=False
)

print("\nFILES SAVED")
print("✓ geographic_state_hotspots.csv")
print("✓ geographic_city_hotspots.csv")

STATE PERFORMANCE — CUSTOMER EXPERIENCE
----------------------------------------------------------------------


,state,total_orders,total_revenue,avg_review_score,negative_review_rate,late_delivery_rate,freight_ratio
25,SP,41746,5202955.05,4.174126,12.589217,5.894555,0.138137
18,RJ,12852,1824092.67,3.877263,20.643178,13.470412,0.167529
10,MG,11635,1585308.03,4.135754,13.259477,5.618670,0.170852
22,RS,5466,750304.02,4.133658,13.301488,7.148204,0.180624
17,PR,5045,683083.76,4.181112,12.412831,4.996953,0.172529
23,SC,3637,520553.34,4.073705,14.657800,9.754722,0.172240
4,BA,3380,511349.99,3.861078,18.922156,14.035627,0.195867
6,DF,2140,302603.94,4.066964,14.849624,7.067308,0.167300
7,ES,2033,275037.31,4.038385,14.905284,12.230576,0.180938
8,GO,2020,294591.95,4.043348,14.698555,8.175779,0.180300



STATE HOTSPOT CANDIDATES
----------------------------------------------------------------------


,state,total_orders,total_revenue,avg_review_score,negative_review_rate,late_delivery_rate,freight_ratio,hotspot_signals
5,CE,1336,227254.71,3.854449,19.683258,15.324472,0.212764,4
13,PA,975,178947.81,3.846154,19.542620,12.367865,0.216260,4
9,MA,747,119648.22,3.758086,21.832884,19.665272,0.263470,4
18,RJ,12852,1824092.67,3.877263,20.643178,13.470412,0.167529,3
4,BA,3380,511349.99,3.861078,18.922156,14.035627,0.195867,3
15,PE,1652,262788.03,4.008563,16.758410,10.797238,0.226227,3
14,PB,536,115268.08,4.016981,16.603774,11.025145,0.223130,3
7,ES,2033,275037.31,4.038385,14.905284,12.230576,0.180938,1
11,MS,715,116812.64,4.109397,14.586255,11.554922,0.163887,1
25,SP,41746,5202955.05,4.174126,12.589217,5.894555,0.138137,0



CITY HOTSPOT CANDIDATES
----------------------------------------------------------------------


...


HIGH-VOLUME CITIES WITH ABOVE-AVERAGE LATE DELIVERY
----------------------------------------------------------------------


,city,total_orders,total_revenue,late_delivery_rate,avg_review_score,negative_review_rate
3155,rio de janeiro,6882,992538.86,11.811024,3.905963,20.361924
707,campinas,1444,187844.53,10.170697,4.092502,13.875263
2964,porto alegre,1379,190562.08,11.782252,4.006565,16.921955
3247,salvador,1245,181104.42,17.508418,3.717391,22.641509
2461,niteroi,849,117907.12,12.484848,3.956057,19.121140
3415,santos,713,98777.09,9.285714,4.198153,12.357955
1445,goiania,692,106111.17,11.515152,3.978676,16.617647
1374,fortaleza,654,97868.06,17.961165,3.811728,21.913580
3086,recife,613,89803.01,13.322091,3.932231,19.338843
1358,florianopolis,570,85925.99,12.746858,4.015071,17.021277



HIGH-VOLUME CITIES WITH WEAKER CUSTOMER EXPERIENCE
----------------------------------------------------------------------


,city,total_orders,total_revenue,avg_review_score,negative_review_rate,late_delivery_rate,freight_ratio
3155,rio de janeiro,6882,992538.86,3.905963,20.361924,11.811024,0.162911
453,belo horizonte,2773,355611.13,4.108751,13.616558,6.154987,0.171879
558,brasilia,2131,301920.25,4.067249,14.818311,7.098020,0.166881
707,campinas,1444,187844.53,4.092502,13.875263,10.170697,0.131477
2964,porto alegre,1379,190562.08,4.006565,16.921955,11.782252,0.175806
3247,salvador,1245,181104.42,3.717391,22.641509,17.508418,0.196947
1529,guarulhos,1189,144268.39,4.077740,14.953271,6.299213,0.133830
2461,niteroi,849,117907.12,3.956057,19.121140,12.484848,0.169729
1445,goiania,692,106111.17,3.978676,16.617647,11.515152,0.169156
1374,fortaleza,654,97868.06,3.811728,21.913580,17.961165,0.212306



FILES SAVED
✓ geographic_state_hotspots.csv
✓ geographic_city_hotspots.csv


In [60]:
# ============================================================
# PHASE 13 — GEOGRAPHIC ANALYSIS
# Cell 3: Final Geographic Evidence Summary
# ============================================================

# ------------------------------------------------------------
# 1. Region comparison
# ------------------------------------------------------------

print("REGION-LEVEL CUSTOMER EXPERIENCE")
print("-" * 75)

region_summary = (
    region_performance[
        [
            "region",
            "total_orders",
            "total_revenue",
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate",
            "freight_ratio"
        ]
    ]
    .sort_values("total_orders", ascending=False)
)

display(region_summary)

# ------------------------------------------------------------
# 2. Highest late-delivery states
# ------------------------------------------------------------

print("\nSTATES WITH HIGHEST LATE-DELIVERY RATES")
print("-" * 75)

display(
    state_performance[
        state_performance["total_orders"] >= 500
    ]
    .sort_values("late_delivery_rate", ascending=False)
    [
        [
            "state",
            "total_orders",
            "late_delivery_rate",
            "avg_review_score",
            "negative_review_rate"
        ]
    ]
    .head(10)
)

# ------------------------------------------------------------
# 3. States with highest negative-review rates
# ------------------------------------------------------------

print("\nSTATES WITH HIGHEST NEGATIVE-REVIEW RATES")
print("-" * 75)

display(
    state_performance[
        state_performance["total_orders"] >= 500
    ]
    .sort_values("negative_review_rate", ascending=False)
    [
        [
            "state",
            "total_orders",
            "negative_review_rate",
            "avg_review_score",
            "late_delivery_rate"
        ]
    ]
    .head(10)
)

# ------------------------------------------------------------
# 4. Geographic hotspot candidates
# ------------------------------------------------------------

print("\nFINAL GEOGRAPHIC HOTSPOT CANDIDATES")
print("-" * 75)

final_hotspots = (
    state_hotspots[
        (state_hotspots["total_orders"] >= 500) &
        (state_hotspots["hotspot_signals"] >= 3)
    ]
    .sort_values(
        ["hotspot_signals", "total_orders"],
        ascending=[False, False]
    )
)

display(
    final_hotspots[
        [
            "state",
            "total_orders",
            "total_revenue",
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate",
            "freight_ratio",
            "hotspot_signals"
        ]
    ]
)

# ------------------------------------------------------------
# 5. Geographic evidence summary for H5
# ------------------------------------------------------------

h5_summary = pd.DataFrame({
    "Hypothesis": [
        "Certain geographic regions have operational problems."
    ],
    "Evidence": [
        "Geographic performance varies across states, cities and regions "
        "across delivery, review and freight metrics."
    ],
    "Assessment": [
        "Descriptive evidence supports further investigation; "
        "geography alone does not establish causation."
    ]
})

print("\nH5 — GEOGRAPHIC HYPOTHESIS EVIDENCE")
print("-" * 75)

display(h5_summary)

# ------------------------------------------------------------
# 6. Save final compact hotspot table
# ------------------------------------------------------------

final_hotspots.to_csv(
    "../data/analytical/geographic_final_hotspots.csv",
    index=False
)

print("\nFINAL FILE SAVED")
print("✓ geographic_final_hotspots.csv")

print("\nPHASE 13 STATUS")
print("-" * 75)
print("✓ State analysis")
print("✓ City analysis")
print("✓ Region analysis")
print("✓ Geographic hotspot screening")
print("✓ H5 evidence summary")
print("✓ Reusable analytical datasets")
print("✓ Final hotspot dataset")
print("\nPHASE 13 COMPLETE")

REGION-LEVEL CUSTOMER EXPERIENCE
---------------------------------------------------------------------------


,region,total_orders,total_revenue,avg_review_score,negative_review_rate,late_delivery_rate,freight_ratio
4,Southeast,68266,8887393.06,4.107941,14.281073,7.451887,0.151330
3,South,14148,1953941.12,4.135207,13.332386,7.050818,0.175560
2,Northeast,9394,1545493.75,3.895936,18.791658,14.329943,0.216957
0,Central-West,5782,870462.06,4.069415,14.596381,7.965861,0.175309
1,North,1851,334353.71,3.956901,17.184943,9.799555,0.227420



STATES WITH HIGHEST LATE-DELIVERY RATES
---------------------------------------------------------------------------


,state,total_orders,late_delivery_rate,avg_review_score,negative_review_rate
9,MA,747,19.665272,3.758086,21.832884
5,CE,1336,15.324472,3.854449,19.683258
4,BA,3380,14.035627,3.861078,18.922156
18,RJ,12852,13.470412,3.877263,20.643178
13,PA,975,12.367865,3.846154,19.542620
7,ES,2033,12.230576,4.038385,14.905284
11,MS,715,11.554922,4.109397,14.586255
14,PB,536,11.025145,4.016981,16.603774
15,PE,1652,10.797238,4.008563,16.758410
23,SC,3637,9.754722,4.073705,14.657800



STATES WITH HIGHEST NEGATIVE-REVIEW RATES
---------------------------------------------------------------------------


...


FINAL GEOGRAPHIC HOTSPOT CANDIDATES
---------------------------------------------------------------------------


,state,total_orders,total_revenue,avg_review_score,negative_review_rate,late_delivery_rate,freight_ratio,hotspot_signals
5,CE,1336,227254.71,3.854449,19.683258,15.324472,0.212764,4
13,PA,975,178947.81,3.846154,19.542620,12.367865,0.216260,4
9,MA,747,119648.22,3.758086,21.832884,19.665272,0.263470,4
18,RJ,12852,1824092.67,3.877263,20.643178,13.470412,0.167529,3
4,BA,3380,511349.99,3.861078,18.922156,14.035627,0.195867,3
15,PE,1652,262788.03,4.008563,16.758410,10.797238,0.226227,3
14,PB,536,115268.08,4.016981,16.603774,11.025145,0.223130,3



H5 — GEOGRAPHIC HYPOTHESIS EVIDENCE
---------------------------------------------------------------------------


,Hypothesis,Evidence,Assessment
0,Certain geographic regions have operational problems.,"Geographic performance varies across states, cities and regions across delivery, review and freight metrics.",Descriptive evidence supports further investigation; geography alone does not establish causation.



FINAL FILE SAVED
✓ geographic_final_hotspots.csv

PHASE 13 STATUS
---------------------------------------------------------------------------
✓ State analysis
✓ City analysis
✓ Region analysis
✓ Geographic hotspot screening
✓ H5 evidence summary
✓ Reusable analytical datasets
✓ Final hotspot dataset

PHASE 13 COMPLETE


# Phase 13 — Geographic Analysis: What We Achieved

## Objective

Analyze customer experience across **states, cities, and macro-regions** to identify geographic areas showing weaker customer-experience signals.

The analysis compares:

- Order volume
- Revenue
- Delivery performance
- Average review score
- Negative-review rate
- Freight ratio

The objective is to identify **geographic hotspot candidates** for further business investigation.

---

## 1. State-Level Analysis

A state-level analytical dataset was created containing:

- Total orders
- Total revenue
- Total freight
- Delivery orders
- Late orders
- Average delivery delay
- Reviewed orders
- Average review score
- Negative-review rate
- Late-delivery rate
- Freight ratio

The analysis showed meaningful variation in customer experience across Brazilian states.

For example, among states with at least 500 orders:

- **Maranhão (MA)** showed a relatively high late-delivery rate of approximately **19.67%** and negative-review rate of approximately **21.83%**.
- **Ceará (CE)** showed approximately **15.32%** late deliveries and **19.68%** negative reviews.
- **Rio de Janeiro (RJ)** had much larger order volume and showed approximately **13.47%** late deliveries and **20.64%** negative reviews.
- **São Paulo (SP)** showed approximately **5.89%** late deliveries and **12.59%** negative reviews.

These differences indicate that customer experience is not uniform across geographic areas.

---

## 2. City-Level Analysis

A city-level analytical dataset was created to investigate more localized patterns.

The analysis identified cities with combinations of:

- Lower-than-average review scores
- Higher-than-average negative-review rates
- Higher-than-average late-delivery rates
- Higher-than-average freight ratios

Examples of cities showing multiple weaker-CX signals included:

- Rio de Janeiro
- Salvador
- Fortaleza
- São Gonçalo
- São Luís
- Maceió
- Teresina

The city-level analysis provides more granular operational signals than state-level analysis.

---

## 3. Region-Level Analysis

Customers were grouped into five macro-regions:

- Southeast
- South
- Northeast
- Central-West
- North

The region-level results showed differences in customer experience.

The **Northeast** recorded approximately:

- Average review score: **3.90**
- Negative-review rate: **18.79%**
- Late-delivery rate: **14.33%**
- Freight ratio: **0.217**

The **Southeast**, which represented the largest order volume, recorded approximately:

- Average review score: **4.11**
- Negative-review rate: **14.28%**
- Late-delivery rate: **7.45%**
- Freight ratio: **0.151**

This indicates that geographic context is associated with substantial differences in observed delivery and satisfaction metrics.

---

## 4. Geographic Hotspot Screening

A hotspot-screening framework was created using four signals:

1. Below-average average review score
2. Above-average negative-review rate
3. Above-average late-delivery rate
4. Above-average freight ratio

Minimum-volume thresholds were applied to reduce the influence of very small geographic groups:

- **State:** minimum 500 orders
- **City:** minimum 100 orders

Areas meeting multiple signals were classified as **hotspot candidates**.

### Important methodological note

These hotspot candidates are **screening results, not statistical proof of geographic causation**.

The signals were generated using relative comparisons against dataset-level averages. They should therefore be interpreted as areas requiring further operational investigation rather than confirmed causes of poor customer experience.

---

## 5. Key Business Finding

Customer experience varies considerably by geography.

Several geographic areas simultaneously show weaker review outcomes, higher negative-review rates, higher late-delivery rates, and/or higher freight burden.

This suggests that **geography may be an important operational segmentation variable** when investigating customer-experience problems.

However, geography itself does not explain why these differences occur.

Potential contributing factors may include:

- Delivery distance
- Seller distribution
- Logistics performance
- Product mix
- Freight burden
- Local operational conditions

These factors must be investigated together before attributing the observed differences to geography alone.

---

## 6. Business Implication

Geographic performance should be incorporated into operational monitoring.

Instead of treating customer experience as a single company-wide metric, management can investigate geographic areas where multiple CX indicators deteriorate simultaneously.

Potential actions include:

- Investigating delivery performance in hotspot candidate areas
- Examining seller concentration within those areas
- Comparing product/category mix
- Reviewing freight burden
- Monitoring changes in review and delivery metrics over time

---

## 7. Output Datasets

The following reusable analytical datasets were created:

- `geographic_state_performance.csv`
- `geographic_city_performance.csv`
- `geographic_region_performance.csv`
- `geographic_state_hotspots.csv`
- `geographic_city_hotspots.csv`
- `geographic_final_hotspots.csv`

---

## Phase 13 Conclusion

**Geographic analysis is complete.**

The analysis successfully examined customer experience at the **state, city, and regional levels** and created a reusable hotspot-screening framework.

### Status

- ✅ State analysis
- ✅ City analysis
- ✅ Region analysis
- ✅ Geographic performance comparison
- ✅ Hotspot candidate screening
- ✅ Reusable analytical datasets
- ✅ Business interpretation
- ✅ Methodological limitations documented

### Key takeaway

> **Customer experience varies geographically, with several areas showing multiple weaker-CX signals. These areas should be treated as investigation candidates rather than assumed geographic root causes.**

In [61]:
# ============================================================
# PHASE 14 — CUSTOMER JOURNEY ANALYSIS
# Step 14.1 — Load Existing Analytical Datasets
# ============================================================

import pandas as pd
import numpy as np

# Existing analytical datasets
order_level = pd.read_csv("../data/analytical/order_level.csv")
seller_performance = pd.read_csv("../data/analytical/seller_performance.csv")
category_performance = pd.read_csv("../data/analytical/category_performance.csv")
geographic_hotspots = pd.read_csv(
    "../data/analytical/geographic_final_hotspots.csv"
)

print("ORDER LEVEL:", order_level.shape)
print("SELLER PERFORMANCE:", seller_performance.shape)
print("CATEGORY PERFORMANCE:", category_performance.shape)
print("GEOGRAPHIC HOTSPOTS:", geographic_hotspots.shape)

ORDER LEVEL: (99441, 19)
SELLER PERFORMANCE: (3095, 16)
CATEGORY PERFORMANCE: (74, 16)
GEOGRAPHIC HOTSPOTS: (7, 18)


In [62]:
# ============================================================
# PHASE 14.2 — CUSTOMER & ORDER STAGE
# ============================================================

# Load the required cleaned datasets
orders_clean = pd.read_csv("../data/cleaned/orders_clean.csv")
customers_clean = pd.read_csv("../data/cleaned/customers_clean.csv")
reviews_clean = pd.read_csv("../data/cleaned/reviews_clean.csv")

# Convert dates
orders_clean["order_purchase_timestamp"] = pd.to_datetime(
    orders_clean["order_purchase_timestamp"],
    errors="coerce"
)

orders_clean["order_delivered_customer_date"] = pd.to_datetime(
    orders_clean["order_delivered_customer_date"],
    errors="coerce"
)

# Keep delivered orders with a valid delivery date
delivered_orders = orders_clean[
    (orders_clean["order_status"] == "delivered") &
    (orders_clean["order_delivered_customer_date"].notna())
].copy()

# Attach review score
review_scores = (
    reviews_clean
    .groupby("order_id", as_index=False)
    .agg(avg_review_score=("review_score", "mean"))
)

delivered_orders = delivered_orders.merge(
    review_scores,
    on="order_id",
    how="left"
)

# Sort by customer and purchase time
delivered_orders = delivered_orders.sort_values(
    ["customer_id", "order_purchase_timestamp"]
).reset_index(drop=True)

# First observed delivered order for each customer
first_orders = (
    delivered_orders
    .groupby("customer_id", as_index=False)
    .first()
)

first_orders = first_orders[
    [
        "customer_id",
        "order_id",
        "order_purchase_timestamp",
        "avg_review_score"
    ]
].rename(
    columns={
        "order_id": "first_order_id",
        "order_purchase_timestamp": "first_order_date",
        "avg_review_score": "first_order_review"
    }
)

# Find the next delivered order for each customer
delivered_orders["next_order_date"] = (
    delivered_orders
    .groupby("customer_id")["order_purchase_timestamp"]
    .shift(-1)
)

# Merge first order with next-order information
journey_customer = first_orders.merge(
    delivered_orders[
        ["customer_id", "order_purchase_timestamp", "next_order_date"]
    ],
    left_on=["customer_id", "first_order_date"],
    right_on=["customer_id", "order_purchase_timestamp"],
    how="left"
)

# Calculate days until next observed order
journey_customer["days_to_next_order"] = (
    journey_customer["next_order_date"]
    - journey_customer["first_order_date"]
).dt.total_seconds() / (60 * 60 * 24)

# Repeat purchase within 90 days
journey_customer["repeat_within_90d"] = (
    journey_customer["days_to_next_order"].between(0, 90, inclusive="both")
)

# Review-based experience grouping
journey_customer["experience_group"] = np.select(
    [
        journey_customer["first_order_review"] <= 2,
        journey_customer["first_order_review"] == 3,
        journey_customer["first_order_review"] >= 4
    ],
    [
        "Negative Experience",
        "Neutral Experience",
        "Positive Experience"
    ],
    default="No Review"
)

print("Customer journey records:", journey_customer.shape)
print()
print("Experience groups:")
print(journey_customer["experience_group"].value_counts(dropna=False))
print()
print("Repeat purchase within 90 days:")
print(journey_customer["repeat_within_90d"].value_counts(dropna=False))

Customer journey records: (96470, 9)

Experience groups:
experience_group
Positive Experience    75619
Negative Experience    12236
Neutral Experience      7915
No Review                700
Name: count, dtype: int64

Repeat purchase within 90 days:
repeat_within_90d
False    96470
Name: count, dtype: int64


In [63]:
# ============================================================
# PHASE 14.2 — CUSTOMER & ORDER STAGE
# CORRECTED REPEAT-PURCHASE ANALYSIS
# ============================================================

# Sort delivered orders by customer and purchase time
delivered_orders = delivered_orders.sort_values(
    ["customer_id", "order_purchase_timestamp"]
).reset_index(drop=True)

# Assign purchase sequence for each customer
delivered_orders["purchase_number"] = (
    delivered_orders.groupby("customer_id").cumcount() + 1
)

# First observed delivered order
first_orders = delivered_orders[
    delivered_orders["purchase_number"] == 1
][
    [
        "customer_id",
        "order_id",
        "order_purchase_timestamp",
        "avg_review_score"
    ]
].copy()

first_orders = first_orders.rename(
    columns={
        "order_id": "first_order_id",
        "order_purchase_timestamp": "first_order_date",
        "avg_review_score": "first_order_review"
    }
)

# Second observed delivered order
second_orders = delivered_orders[
    delivered_orders["purchase_number"] == 2
][
    [
        "customer_id",
        "order_purchase_timestamp"
    ]
].copy()

second_orders = second_orders.rename(
    columns={
        "order_purchase_timestamp": "second_order_date"
    }
)

# Join first and second orders
journey_customer = first_orders.merge(
    second_orders,
    on="customer_id",
    how="left"
)

# Calculate time to second observed purchase
journey_customer["days_to_second_order"] = (
    journey_customer["second_order_date"]
    - journey_customer["first_order_date"]
).dt.total_seconds() / (60 * 60 * 24)

# Repeat purchase within 90 days
journey_customer["repeat_within_90d"] = (
    journey_customer["days_to_second_order"].between(
        0, 90, inclusive="both"
    )
)

# Experience grouping
journey_customer["experience_group"] = np.select(
    [
        journey_customer["first_order_review"] <= 2,
        journey_customer["first_order_review"] == 3,
        journey_customer["first_order_review"] >= 4
    ],
    [
        "Negative Experience",
        "Neutral Experience",
        "Positive Experience"
    ],
    default="No Review"
)

print("Customer journey records:", journey_customer.shape)

print("\nExperience groups:")
print(
    journey_customer["experience_group"]
    .value_counts(dropna=False)
)

print("\nCustomers with a second delivered order:")
print(
    journey_customer["second_order_date"]
    .notna()
    .value_counts()
)

print("\nRepeat purchase within 90 days:")
print(
    journey_customer["repeat_within_90d"]
    .value_counts(dropna=False)
)

Customer journey records: (96470, 8)

Experience groups:
experience_group
Positive Experience    75619
Negative Experience    12236
Neutral Experience      7915
No Review                700
Name: count, dtype: int64

Customers with a second delivered order:
second_order_date
False    96470
Name: count, dtype: int64

Repeat purchase within 90 days:
repeat_within_90d
False    96470
Name: count, dtype: int64


In [64]:
# ============================================================
# PHASE 14.2 — DIAGNOSTIC: CUSTOMER ID RELATIONSHIP
# ============================================================

print("Orders:")
print("Total orders:", len(orders_clean))
print("Unique customer_id:", orders_clean["customer_id"].nunique())

print("\nCustomers:")
print("Total customer records:", len(customers_clean))
print("Unique customer_id:", customers_clean["customer_id"].nunique())
print("Unique customer_unique_id:", customers_clean["customer_unique_id"].nunique())

# How many orders are associated with each real customer?
customer_order_counts = (
    orders_clean
    .merge(
        customers_clean[
            ["customer_id", "customer_unique_id"]
        ],
        on="customer_id",
        how="left"
    )
    .groupby("customer_unique_id")
    .size()
)

print("\nOrders per real customer:")
print(customer_order_counts.describe())

print("\nCustomers with more than 1 order:")
print(
    (customer_order_counts > 1).sum()
)

print("\nCustomers with 2+ orders:")
print(
    customer_order_counts[
        customer_order_counts > 1
    ].head(20)
)

Orders:
Total orders: 99441
Unique customer_id: 99441

Customers:
Total customer records: 99441
Unique customer_id: 99441
Unique customer_unique_id: 96096

Orders per real customer:
count    96096.000000
mean         1.034809
std          0.214384
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000
dtype: float64

Customers with more than 1 order:
2997

Customers with 2+ orders:
customer_unique_id
00172711b30d52eea8b313a7f2cced02    2
004288347e5e88a27ded2bb23747066c    2
004b45ec5c64187465168251cd1c9c2f    2
0058f300f57d7b93c477a131a59b36c3    2
00a39521eb40f7012db50455bf083460    2
00cc12a6d8b578b8ebd21ea4e2ae8b27    2
011575986092c30523ecb71ff10cb473    2
011b4adcd54683b480c4d841250a987f    2
012452d40dafae4df401bced74cdb490    2
012a218df8995d3ec3bb221828360c86    2
013ef03e0f3f408dd9bf555e4edcdc0a    2
013f4353d26bb05dc6652f1269458d8d    2
015557c9912277312b9073947804a7ba    2
0178b244a5c281fb2ade54038dd4b161    2
01886ef98

In [65]:
# ============================================================
# PHASE 14.2 — CUSTOMER & ORDER STAGE
# FINAL CUSTOMER-LEVEL ANALYSIS
# ============================================================

# ------------------------------------------------------------
# 1. Attach real customer identity to every order
# ------------------------------------------------------------

customer_orders = orders_clean.merge(
    customers_clean[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id",
    how="left"
)

print(
    "Missing customer_unique_id:",
    customer_orders["customer_unique_id"].isna().sum()
)

# ------------------------------------------------------------
# 2. Convert purchase and delivery dates
# ------------------------------------------------------------

customer_orders["order_purchase_timestamp"] = pd.to_datetime(
    customer_orders["order_purchase_timestamp"],
    errors="coerce"
)

customer_orders["order_delivered_customer_date"] = pd.to_datetime(
    customer_orders["order_delivered_customer_date"],
    errors="coerce"
)

# ------------------------------------------------------------
# 3. Keep delivered orders with a valid delivery date
# ------------------------------------------------------------

delivered_customer_orders = customer_orders[
    (customer_orders["order_status"] == "delivered") &
    (customer_orders["order_delivered_customer_date"].notna())
].copy()

# ------------------------------------------------------------
# 4. Attach review scores
# ------------------------------------------------------------

review_scores = (
    reviews_clean
    .groupby("order_id", as_index=False)
    .agg(
        first_review_score=("review_score", "mean")
    )
)

delivered_customer_orders = delivered_customer_orders.merge(
    review_scores,
    on="order_id",
    how="left"
)

# ------------------------------------------------------------
# 5. Sort by REAL customer and purchase date
# ------------------------------------------------------------

delivered_customer_orders = (
    delivered_customer_orders
    .sort_values(
        ["customer_unique_id", "order_purchase_timestamp"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. Assign purchase sequence
# ------------------------------------------------------------

delivered_customer_orders["purchase_number"] = (
    delivered_customer_orders
    .groupby("customer_unique_id")
    .cumcount() + 1
)

# ------------------------------------------------------------
# 7. First delivered order
# ------------------------------------------------------------

first_orders = delivered_customer_orders[
    delivered_customer_orders["purchase_number"] == 1
][
    [
        "customer_unique_id",
        "order_id",
        "order_purchase_timestamp",
        "first_review_score"
    ]
].copy()

first_orders = first_orders.rename(
    columns={
        "order_id": "first_order_id",
        "order_purchase_timestamp": "first_order_date",
        "first_review_score": "first_order_review"
    }
)

# ------------------------------------------------------------
# 8. Second delivered order
# ------------------------------------------------------------

second_orders = delivered_customer_orders[
    delivered_customer_orders["purchase_number"] == 2
][
    [
        "customer_unique_id",
        "order_purchase_timestamp"
    ]
].copy()

second_orders = second_orders.rename(
    columns={
        "order_purchase_timestamp": "second_order_date"
    }
)

# ------------------------------------------------------------
# 9. Join first and second order
# ------------------------------------------------------------

journey_customer = first_orders.merge(
    second_orders,
    on="customer_unique_id",
    how="left"
)

# ------------------------------------------------------------
# 10. Calculate time to second purchase
# ------------------------------------------------------------

journey_customer["days_to_second_order"] = (
    journey_customer["second_order_date"]
    - journey_customer["first_order_date"]
).dt.total_seconds() / (60 * 60 * 24)

# ------------------------------------------------------------
# 11. Repeat purchase within 90 days
# ------------------------------------------------------------

journey_customer["repeat_within_90d"] = (
    journey_customer["days_to_second_order"]
    .between(0, 90, inclusive="both")
)

# Customers without a second order should NOT be treated as
# having a measured 90-day repeat outcome unless their
# observation window is fully observable.
#
# For now we retain the flag and will handle the observation
# window explicitly in the next step.

# ------------------------------------------------------------
# 12. Experience grouping
# ------------------------------------------------------------

journey_customer["experience_group"] = np.select(
    [
        journey_customer["first_order_review"] <= 2,
        journey_customer["first_order_review"] == 3,
        journey_customer["first_order_review"] >= 4
    ],
    [
        "Negative Experience",
        "Neutral Experience",
        "Positive Experience"
    ],
    default="No Review"
)

# ------------------------------------------------------------
# 13. Audit
# ------------------------------------------------------------

print("\nCustomer journey records:", journey_customer.shape)

print("\nExperience groups:")
print(
    journey_customer["experience_group"]
    .value_counts(dropna=False)
)

print("\nCustomers with a second delivered order:")
print(
    journey_customer["second_order_date"]
    .notna()
    .value_counts()
)

print("\nDays to second order:")
print(
    journey_customer["days_to_second_order"]
    .describe()
)

Missing customer_unique_id: 0

Customer journey records: (93350, 8)

Experience groups:
experience_group
Positive Experience    73144
Negative Experience    11895
Neutral Experience      7665
No Review                646
Name: count, dtype: int64

Customers with a second delivered order:
second_order_date
False    90549
True      2801
Name: count, dtype: int64

Days to second order:
count    2801.000000
mean       81.206685
std       110.004017
min         0.000000
25%         0.002130
50%        28.980197
75%       126.263588
max       608.978912
Name: days_to_second_order, dtype: float64


In [66]:
# ============================================================
# PHASE 14.2 — 90-DAY OBSERVATION WINDOW
# ============================================================

# Dataset end date based on the last observed purchase date
dataset_end_date = customer_orders[
    "order_purchase_timestamp"
].max()

print("Dataset end date:", dataset_end_date)

# A customer can only have a fully observable 90-day window
# if their first delivered order occurred at least 90 days
# before the dataset ended.
journey_customer["fully_observable_90d"] = (
    journey_customer["first_order_date"]
    <= dataset_end_date - pd.Timedelta(days=90)
)

# Keep only customers with a complete 90-day observation window
journey_90d = journey_customer[
    journey_customer["fully_observable_90d"]
].copy()

print("\nFully observable 90-day customers:", journey_90d.shape[0])

print("\nExperience groups:")
print(
    journey_90d["experience_group"]
    .value_counts(dropna=False)
)

print("\nRepeat within 90 days:")
print(
    journey_90d["repeat_within_90d"]
    .value_counts(dropna=False)
)

Dataset end date: 2018-10-17 17:30:18

Fully observable 90-day customers: 84371

Experience groups:
experience_group
Positive Experience    65666
Negative Experience    11032
Neutral Experience      7065
No Review                608
Name: count, dtype: int64

Repeat within 90 days:
repeat_within_90d
False    82534
True      1837
Name: count, dtype: int64


In [67]:
# ============================================================
# PHASE 14.2 — REPEAT PURCHASE RATE BY EXPERIENCE
# ============================================================

repeat_summary = (
    journey_90d
    .groupby("experience_group")
    .agg(
        customers=("customer_unique_id", "count"),
        repeat_customers=("repeat_within_90d", "sum")
    )
    .reset_index()
)

repeat_summary["repeat_rate_90d"] = (
    repeat_summary["repeat_customers"]
    / repeat_summary["customers"]
    * 100
)

repeat_summary = repeat_summary.sort_values(
    "repeat_rate_90d",
    ascending=False
)

print(repeat_summary.to_string(index=False))

   experience_group  customers  repeat_customers  repeat_rate_90d
          No Review        608                35         5.756579
Positive Experience      65666              1417         2.157890
Negative Experience      11032               236         2.139231
 Neutral Experience       7065               149         2.108988


In [68]:
# ============================================================
# PHASE 14.2 — CUSTOMER STAGE: BUSINESS INTERPRETATION
# ============================================================

print("\nCustomer & Order Stage Summary")
print("-" * 60)

for _, row in repeat_summary.iterrows():
    print(
        f"{row['experience_group']}: "
        f"{row['repeat_rate_90d']:.2f}% "
        f"repeat within 90 days "
        f"({int(row['repeat_customers'])} of "
        f"{int(row['customers'])} customers)"
    )


Customer & Order Stage Summary
------------------------------------------------------------
No Review: 5.76% repeat within 90 days (35 of 608 customers)
Positive Experience: 2.16% repeat within 90 days (1417 of 65666 customers)
Negative Experience: 2.14% repeat within 90 days (236 of 11032 customers)
Neutral Experience: 2.11% repeat within 90 days (149 of 7065 customers)


In [69]:
# ============================================================
# PHASE 14.3 — SELLER STAGE
# ============================================================

print("SELLER STAGE ANALYSIS")
print("=" * 60)

# Sellers with sufficient review volume
seller_stage = seller_performance[
    seller_performance["reviewed_orders"] >= 30
].copy()

print("Sellers with >=30 reviewed orders:", len(seller_stage))

print("\nAverage seller metrics:")
print(
    seller_stage[
        [
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate"
        ]
    ].mean()
)

print("\nSeller performance ranges:")
print(
    seller_stage[
        [
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate"
        ]
    ].describe()
)

# ------------------------------------------------------------
# High-revenue sellers with weaker customer experience
# ------------------------------------------------------------

high_revenue_lower_experience = seller_stage[
    seller_stage["seller_segment"]
    == "High Revenue + Lower Experience"
].copy()

print("\nHigh Revenue + Lower Experience sellers:",
      len(high_revenue_lower_experience))

print("\nSegment summary:")
print(
    high_revenue_lower_experience[
        [
            "total_orders",
            "total_revenue",
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate"
        ]
    ].mean()
)

# ------------------------------------------------------------
# Compare with high-revenue / high-experience sellers
# ------------------------------------------------------------

high_revenue_high_experience = seller_stage[
    seller_stage["seller_segment"]
    == "High Revenue + High Experience"
].copy()

print("\nHigh Revenue + High Experience sellers:",
      len(high_revenue_high_experience))

print("\nComparison:")
comparison = pd.DataFrame({
    "High Revenue + Lower Experience": [
        high_revenue_lower_experience["avg_review_score"].mean(),
        high_revenue_lower_experience["negative_review_rate"].mean(),
        high_revenue_lower_experience["late_delivery_rate"].mean()
    ],
    "High Revenue + High Experience": [
        high_revenue_high_experience["avg_review_score"].mean(),
        high_revenue_high_experience["negative_review_rate"].mean(),
        high_revenue_high_experience["late_delivery_rate"].mean()
    ]
}, index=[
    "Average Review Score",
    "Negative Review Rate",
    "Late Delivery Rate"
])

print(comparison)

SELLER STAGE ANALYSIS
Sellers with >=30 reviewed orders: 628

Average seller metrics:
avg_review_score         4.116574
negative_review_rate    14.112660
late_delivery_rate       7.762498
dtype: float64

Seller performance ranges:
       avg_review_score  negative_review_rate  late_delivery_rate
count        628.000000            628.000000          628.000000
mean           4.116574             14.112660            7.762498
std            0.314363              7.623207            5.446204
min            2.333333              0.000000            0.000000
25%            3.944220              8.918651            3.560924
50%            4.153796             13.043478            6.557377
75%            4.325383             18.032787           10.467885
max            5.000000             61.403509           34.883721

High Revenue + Lower Experience sellers: 314

Segment summary:
total_orders              150.859873
total_revenue           17405.364522
avg_review_score            3.882989


In [70]:
# ============================================================
# PHASE 14.4 — PRODUCT STAGE
# ============================================================

print("PRODUCT STAGE ANALYSIS")
print("=" * 60)

# Exclude the unclassified/missing category from
# category-to-category comparison
product_stage = category_performance[
    category_performance["category"] != "Unclassified / Missing Category"
].copy()

print("Product categories analyzed:", len(product_stage))

# ------------------------------------------------------------
# Overall category performance
# ------------------------------------------------------------

print("\nAverage category metrics:")
print(
    product_stage[
        [
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate",
            "freight_ratio"
        ]
    ].mean()
)

# ------------------------------------------------------------
# Categories with strongest customer-experience concerns
# ------------------------------------------------------------

print("\nLowest-rated categories:")
print(
    product_stage[
        [
            "category",
            "total_orders",
            "total_revenue",
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate"
        ]
    ]
    .sort_values("avg_review_score")
    .head(10)
    .to_string(index=False)
)

# ------------------------------------------------------------
# High-revenue + lower-experience categories
# ------------------------------------------------------------

high_revenue_lower_experience_categories = product_stage[
    product_stage["category_segment"]
    == "High Revenue + Lower Experience"
].copy()

print(
    "\nHigh Revenue + Lower Experience categories:",
    len(high_revenue_lower_experience_categories)
)

print("\nSegment:")
print(
    high_revenue_lower_experience_categories[
        [
            "category",
            "total_orders",
            "total_revenue",
            "avg_review_score",
            "negative_review_rate",
            "late_delivery_rate",
            "freight_ratio"
        ]
    ]
    .sort_values("avg_review_score")
    .to_string(index=False)
)

PRODUCT STAGE ANALYSIS
Product categories analyzed: 73

Average category metrics:
avg_review_score         4.071249
negative_review_rate    15.264466
late_delivery_rate       7.064497
freight_ratio            0.193159
dtype: float64

Lowest-rated categories:
                                     category  total_orders  total_revenue  avg_review_score  negative_review_rate  late_delivery_rate
                        security_and_services             2         283.29          2.500000             50.000000            0.000000
                                     pc_gamer             8        1545.95          3.125000             37.500000            0.000000
portateis_cozinha_e_preparadores_de_alimentos            14        3968.53          3.428571             28.571429            7.692308
                             office_furniture          1273      273960.70          3.617508             22.802850            9.170654
                        fashion_male_clothing           112       

In [71]:
# ============================================================
# PHASE 14.5 — END-TO-END CUSTOMER JOURNEY
# CUSTOMER → ORDER → SELLER → PRODUCT → DELIVERY → REVIEW
# ============================================================

print("END-TO-END CUSTOMER JOURNEY")
print("=" * 60)

# ------------------------------------------------------------
# 1. Start from order-level data
# ------------------------------------------------------------

journey_orders = order_level[
    [
        "order_id",
        "customer_id",
        "order_status",
        "delivery_delay_days",
        "is_late",
        "avg_review_score"
    ]
].copy()

# ------------------------------------------------------------
# 2. Add customer identity
# ------------------------------------------------------------

journey_orders = journey_orders.merge(
    customers_clean[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id",
    how="left"
)

# ------------------------------------------------------------
# 3. Add seller information
# ------------------------------------------------------------

order_seller = (
    order_items_clean
    .groupby("order_id")
    .agg(
        seller_count=("seller_id", "nunique")
    )
    .reset_index()
)

journey_orders = journey_orders.merge(
    order_seller,
    on="order_id",
    how="left"
)

# ------------------------------------------------------------
# 4. Add product/category information
# ------------------------------------------------------------

order_category = (
    order_item_level
    .groupby("order_id")
    .agg(
        product_category_count=("category", "nunique")
    )
    .reset_index()
)

journey_orders = journey_orders.merge(
    order_category,
    on="order_id",
    how="left"
)

# ------------------------------------------------------------
# 5. Create experience classification
# ------------------------------------------------------------

journey_orders["experience_group"] = np.select(
    [
        journey_orders["avg_review_score"] <= 2,
        journey_orders["avg_review_score"] == 3,
        journey_orders["avg_review_score"] >= 4
    ],
    [
        "Negative Experience",
        "Neutral Experience",
        "Positive Experience"
    ],
    default="No Review"
)

# ------------------------------------------------------------
# 6. Journey-stage summary
# ------------------------------------------------------------

journey_stage_summary = pd.DataFrame({
    "Stage": [
        "Customer / Order",
        "Seller",
        "Product",
        "Delivery",
        "Review"
    ],
    "What_we_measure": [
        "Repeat purchase within 90 days",
        "Seller review and delivery performance",
        "Category satisfaction and negative reviews",
        "Delivery delay and lateness",
        "Customer review score and complaint themes"
    ]
})

print(journey_stage_summary.to_string(index=False))

# ------------------------------------------------------------
# 7. Overall journey data quality
# ------------------------------------------------------------

print("\nJourney dataset:", journey_orders.shape)
print("Unique orders:", journey_orders["order_id"].nunique())
print(
    "Missing customer identity:",
    journey_orders["customer_unique_id"].isna().sum()
)
print(
    "Missing seller information:",
    journey_orders["seller_count"].isna().sum()
)
print(
    "Missing product information:",
    journey_orders["product_category_count"].isna().sum()
)
print(
    "Missing delivery delay:",
    journey_orders["delivery_delay_days"].isna().sum()
)
print(
    "Missing review score:",
    journey_orders["avg_review_score"].isna().sum()
)

END-TO-END CUSTOMER JOURNEY


NameError: name 'order_item_level' is not defined

In [ ]:
# ============================================================
# PHASE 14.5 — FIX PRODUCT CATEGORY LINKAGE
# ============================================================

# Load the cleaned files needed for the product stage
order_items_clean = pd.read_csv(
    "../data/cleaned/order_items_clean.csv"
)

products_clean = pd.read_csv(
    "../data/cleaned/products_clean.csv"
)

# ------------------------------------------------------------
# Create order-level product/category information
# ------------------------------------------------------------

order_category = (
    order_items_clean
    .merge(
        products_clean[
            ["product_id", "product_category_name"]
        ],
        on="product_id",
        how="left"
    )
    .groupby("order_id")
    .agg(
        product_category_count=(
            "product_category_name",
            "nunique"
        )
    )
    .reset_index()
)

print("Order-category records:", len(order_category))
print(
    "Unique orders:",
    order_category["order_id"].nunique()
)

print(
    "Duplicate order IDs:",
    order_category["order_id"].duplicated().sum()
)

In [ ]:
# ============================================================
# PHASE 14.5 — END-TO-END CUSTOMER JOURNEY
# CONTINUED
# ============================================================

# ------------------------------------------------------------
# 1. Start from order-level data
# ------------------------------------------------------------

journey_orders = order_level[
    [
        "order_id",
        "customer_id",
        "order_status",
        "delivery_delay_days",
        "is_late",
        "avg_review_score"
    ]
].copy()

# ------------------------------------------------------------
# 2. Add real customer identity
# ------------------------------------------------------------

journey_orders = journey_orders.merge(
    customers_clean[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id",
    how="left"
)

# ------------------------------------------------------------
# 3. Add seller information
# ------------------------------------------------------------

order_seller = (
    order_items_clean
    .groupby("order_id")
    .agg(
        seller_count=("seller_id", "nunique")
    )
    .reset_index()
)

journey_orders = journey_orders.merge(
    order_seller,
    on="order_id",
    how="left"
)

# ------------------------------------------------------------
# 4. Add product/category information
# ------------------------------------------------------------

journey_orders = journey_orders.merge(
    order_category,
    on="order_id",
    how="left"
)

# ------------------------------------------------------------
# 5. Create experience classification
# ------------------------------------------------------------

journey_orders["experience_group"] = np.select(
    [
        journey_orders["avg_review_score"] <= 2,
        journey_orders["avg_review_score"] == 3,
        journey_orders["avg_review_score"] >= 4
    ],
    [
        "Negative Experience",
        "Neutral Experience",
        "Positive Experience"
    ],
    default="No Review"
)

# ------------------------------------------------------------
# 6. Journey-stage summary
# ------------------------------------------------------------

journey_stage_summary = pd.DataFrame({
    "Stage": [
        "Customer / Order",
        "Seller",
        "Product",
        "Delivery",
        "Review"
    ],
    "What_we_measure": [
        "Repeat purchase within 90 days",
        "Seller review and delivery performance",
        "Category satisfaction and negative reviews",
        "Delivery delay and lateness",
        "Customer review score and complaint themes"
    ]
})

print(journey_stage_summary.to_string(index=False))

# ------------------------------------------------------------
# 7. End-to-end data quality audit
# ------------------------------------------------------------

print("\nJourney dataset:", journey_orders.shape)

print(
    "Unique orders:",
    journey_orders["order_id"].nunique()
)

print(
    "Duplicate orders:",
    journey_orders["order_id"].duplicated().sum()
)

print(
    "Missing customer identity:",
    journey_orders["customer_unique_id"].isna().sum()
)

print(
    "Missing seller information:",
    journey_orders["seller_count"].isna().sum()
)

print(
    "Missing product information:",
    journey_orders["product_category_count"].isna().sum()
)

print(
    "Missing delivery delay:",
    journey_orders["delivery_delay_days"].isna().sum()
)

print(
    "Missing review score:",
    journey_orders["avg_review_score"].isna().sum()
)

# Phase 14 — Customer Journey Analysis

## Objective

Trace the customer experience across the complete journey:

**CUSTOMER → ORDER → SELLER → PRODUCT → DELIVERY → REVIEW**

The objective was to identify where customer experience appears to break down and connect customer-level, seller-level, product-level, delivery-level and review-level evidence.

---

## 1. Customer / Order Stage

Customer identity was analyzed using `customer_unique_id`, which represents the real customer across multiple order-level `customer_id` records.

A 90-day repeat-purchase analysis was performed using customers whose first delivered order occurred at least 90 days before the dataset end date, ensuring a complete observation window.

### Result

- Fully observable customers: **84,371**
- Repeat customers within 90 days: **1,837**
- Customers without a repeat within 90 days: **82,534**

### Repeat Purchase by First Experience

| First Experience | Customers | Repeat Customers | 90-Day Repeat Rate |
|---|---:|---:|---:|
| Positive | 65,666 | 1,417 | 2.16% |
| Negative | 11,032 | 236 | 2.14% |
| Neutral | 7,065 | 149 | 2.11% |
| No Review | 608 | 35 | 5.76% |

Positive, neutral and negative reviewed customers showed very similar 90-day repeat-purchase rates.

Therefore, first-order review experience was not treated as a strong standalone indicator of 90-day repeat purchasing in this dataset.

The `No Review` group was retained for completeness but was not interpreted as evidence of higher customer loyalty because of its small sample size and different review behavior.

### Limitation

This analysis measures repeat purchasing within a 90-day window and does not establish long-term retention or causality.

---

## 2. Seller Stage

Seller performance was evaluated using sellers with at least **30 reviewed orders** to reduce instability from very small review samples.

### Seller Performance

Among 628 sellers meeting the review-volume threshold:

- Average review score: **4.12**
- Average negative-review rate: **14.11%**
- Average late-delivery rate: **7.76%**

Seller performance varied considerably across the population.

### High-Revenue Seller Comparison

High-revenue sellers were segmented into lower- and higher-experience groups.

| Metric | High Revenue + Lower Experience | High Revenue + High Experience |
|---|---:|---:|
| Sellers | 314 | 312 |
| Average Review Score | 3.88 | 4.35 |
| Negative Review Rate | 19.34% | 8.94% |
| Late Delivery Rate | 9.91% | 5.64% |

The lower-experience high-revenue seller segment therefore shows substantially higher negative-review and late-delivery rates.

This identifies seller execution as an important area for operational investigation.

### Limitation

These are observational segment comparisons. They do not establish that seller performance alone causes lower customer satisfaction.

---

## 3. Product Stage

Product performance was evaluated across **73 product categories** after excluding the unclassified/missing category from category-to-category comparison.

Category performance varied substantially.

Low-volume categories were not treated as major business findings because small sample sizes can produce unstable averages.

### High-Revenue + Lower-Experience Categories

Several categories combined meaningful revenue with relatively weaker customer-experience metrics.

Notable examples include:

- **office_furniture** — average review **3.62**, negative-review rate **22.80%**
- **bed_bath_table** — average review **3.97**, negative-review rate **17.04%**
- **furniture_decor** — average review **4.01**, negative-review rate **16.90%**
- **computers_accessories** — average review **4.03**, negative-review rate **16.21%**
- **baby** — average review **4.04**, negative-review rate **16.01%**

`office_furniture` showed the lowest average review score and highest negative-review rate within this high-revenue/lower-experience category segment.

Product category performance therefore provides another useful operational segmentation.

---

## 4. Delivery Stage

Delivery was one of the strongest customer-experience signals identified in the project.

Earlier statistical analysis found significant differences in review-score distributions across delivery-delay severity groups.

Kruskal-Wallis:

- H = **8637.20**
- p < **0.001**
- n = **95,824**
- Epsilon-squared = **0.0901**

Median review scores declined across increasing delivery-delay severity.

This provides strong evidence of an association between delivery timeliness and customer satisfaction.

### Limitation

The analysis demonstrates association rather than causation. Other factors may contribute to both delivery performance and customer satisfaction.

---

## 5. Review Stage

Customer reviews were analyzed using both structured ratings and written review content.

Star ratings were used as the primary quantitative satisfaction measure.

Written reviews were used to identify customer complaint themes and provide qualitative context.

An AI sentiment model was tested on a balanced validation sample of 250 reviews.

The model agreed with the rating-derived sentiment baseline on **57.2%** of reviews.

Because the model showed substantial disagreement, particularly for fulfillment-related complaints, its output was not treated as ground truth or used as the primary full-dataset sentiment label.

Instead:

- Star ratings → quantitative satisfaction
- Written reviews → complaint/theme analysis
- AI → hypothesis and classification support
- Structured Olist data → validation

---

## 6. End-to-End Journey Dataset

The final integrated journey dataset contains:

- **99,441 orders**
- **0 duplicate orders**
- **0 missing customer identities**
- **775 orders without seller/product information**
- **2,965 orders without delivery-delay information**
- **768 orders without review scores**

The missing seller/product records correspond to orders without associated order-item records and were retained rather than artificially removed.

Missing delivery and review information was also retained because these fields are legitimately unavailable for some orders.

---

## Customer Journey Finding

The customer journey analysis suggests that customer-experience variation is concentrated more strongly around **delivery execution, seller performance and specific product categories** than around first-order review score as a standalone predictor of 90-day repeat purchase.

Delivery performance showed the strongest statistical association with satisfaction among the analyzed operational factors.

Seller-level comparisons also showed substantial differences in customer-experience and delivery metrics, while selected product categories showed weaker satisfaction and higher negative-review rates.

The repeat-purchase analysis did not show a meaningful difference between positive, neutral and negative first-order review groups within the observed 90-day window.

### Business Implication

Customer-experience improvement should therefore focus on operational execution — particularly delivery performance, seller execution and category-specific problems — rather than assuming that customer retention can be explained by review score alone.

### Limitation

All findings are based on observational historical marketplace data. Associations should not be interpreted as causal effects.

---

## Status

**PHASE 14 COMPLETE**

In [ ]:
# ============================================================
# PHASE 15.1 — SAVE POWER BI SUPPORT DATASETS
# ============================================================

# Save customer-level journey analysis
journey_90d.to_csv(
    "../data/analytical/customer_journey_90d.csv",
    index=False
)

# Save integrated order-level journey dataset
journey_orders.to_csv(
    "../data/analytical/customer_journey_orders.csv",
    index=False
)

print("Power BI support datasets saved successfully.")

print("\nFiles:")
print("1. customer_journey_90d.csv")
print("2. customer_journey_orders.csv")

print("\nShapes:")
print(
    "customer_journey_90d:",
    journey_90d.shape
)

print(
    "customer_journey_orders:",
    journey_orders.shape
)

In [81]:
import re
import pandas as pd

# Load cleaned reviews
reviews_clean = pd.read_csv("../data/cleaned/reviews_clean.csv")

# Build review text
reviews_text = reviews_clean[
    ["order_id", "review_id", "review_score",
     "review_comment_title", "review_comment_message"]
].copy()

reviews_text["review_text"] = (
    reviews_text["review_comment_title"].fillna("").astype(str).str.strip()
    + " "
    + reviews_text["review_comment_message"].fillna("").astype(str).str.strip()
).str.strip()

# Keep reviews with meaningful alphabetic text
nlp_reviews = reviews_text[
    reviews_text["review_text"].str.count(r"[A-Za-zÀ-ÿ]") >= 3
].copy()

# Recreate the exact 300-review sample
complaint_sample = (
    nlp_reviews[nlp_reviews["review_score"] <= 3]
    .sample(n=300, random_state=42)
    .reset_index(drop=True)
)

# Exact complaint taxonomy rules from the notebook
complaint_patterns = {

    "Late Delivery": [
        r"\batras", r"\batraso", r"\batrasada", r"\batrasado",
        r"\bdemora", r"\bdemorou", r"\btarde", r"\bprazo.*entrega",
        r"\bentrega.*prazo", r"\bchegou.*tarde", r"\bchegou.*atras"
    ],

    "Product Not Received": [
        r"\bn[aã]o.*cheg", r"\bnunca.*cheg", r"\bn[aã]o.*receb",
        r"\bn[aã]o.*entreg", r"\bproduto.*n[aã]o.*cheg",
        r"\bpedido.*n[aã]o.*cheg", r"\bpedido.*n[aã]o.*receb"
    ],

    "Missing Product / Item": [
        r"\bfaltou", r"\bfaltando", r"\bfaltaram", r"\bitens?.*falt",
        r"\bproduto.*falt", r"\bveio.*falt", r"\bn[aã]o.*veio"
    ],

    "Wrong Product": [
        r"\bproduto.*errad", r"\bveio.*errad", r"\bpedido.*errad",
        r"\breceb.*errad", r"\bdiferente.*an[uú]ncio",
        r"\bn[aã]o.*foi.*comprad"
    ],

    "Product Damage / Defect": [
        r"\bdefeit", r"\bquebrad", r"\bdanific", r"\bdanificad",
        r"\bdanificado", r"\bavari", r"\bn[aã]o.*funcion",
        r"\bparou.*funcion", r"\bdefeituos"
    ],

    "Product Quality": [
        r"\bqualidade", r"\bqualidad.*ruim", r"\bfr[aá]gil",
        r"\bpot[eê]ncia", r"\bmaterial.*ruim", r"\bacabamento",
        r"\bpior.*esper", r"\bbaixa.*qualidade"
    ],

    "Seller / Service Issue": [
        r"\bvendedor", r"\bloja", r"\batendimento", r"\bsuporte",
        r"\bcontato", r"\bresposta", r"\bresponder", r"\bempresa.*n[aã]o",
        r"\bsem.*resposta"
    ],

    "Packaging": [
        r"\bembalagem", r"\bembalado", r"\bcaixa.*dan",
        r"\bcaixa.*queb", r"\bmal.*embalad"
    ],

    "Shipping / Logistics": [
        r"\btransportadora", r"\btransport", r"\brastreio",
        r"\brastreamento", r"\bcorreio", r"\bfrete",
        r"\blog[ií]stic"
    ],

    "Payment": [
        r"\bpagamento", r"\bcobran[cç]a", r"\bcart[aã]o",
        r"\bparcel", r"\bpag[ouo]", r"\bvalor.*cobr"
    ],

    "Cancellation / Refund": [
        r"\bcancel", r"\breembolso", r"\bdevolu[cç]",
        r"\bdinheiro.*volta", r"\bestorno"
    ]
}

def classify_keywords(text):
    text = str(text).lower()

    matched_categories = []

    for category, patterns in complaint_patterns.items():
        if any(re.search(pattern, text) for pattern in patterns):
            matched_categories.append(category)

    return matched_categories


# Classify the 300 reviews
complaint_sample["keyword_categories"] = (
    complaint_sample["review_text"]
    .apply(classify_keywords)
)

complaint_sample["keyword_is_complaint"] = (
    complaint_sample["keyword_categories"].str.len() > 0
)

# Create Power BI dataset
complaint_analysis = complaint_sample[
    [
        "review_id",
        "order_id",
        "review_score",
        "review_text",
        "keyword_is_complaint",
        "keyword_categories"
    ]
].copy()

# One row per review-category combination
complaint_analysis = (
    complaint_analysis
    .explode("keyword_categories")
    .reset_index(drop=True)
)

complaint_analysis["complaint_category"] = (
    complaint_analysis["keyword_categories"]
    .fillna("No Clear Complaint")
)

complaint_analysis = complaint_analysis.drop(
    columns=["keyword_categories"]
)

# Save
complaint_analysis.to_csv(
    "../data/analytical/complaint_analysis.csv",
    index=False
)

print("COMPLAINT DATASET CREATED")
print("-------------------------")
print("Reviews:", complaint_sample["review_id"].nunique())
print("Rows after category expansion:", len(complaint_analysis))
print("\nPotential complaints:",
      complaint_sample["keyword_is_complaint"].sum())
print("No keyword match:",
      (~complaint_sample["keyword_is_complaint"]).sum())

print("\nCategory counts:")
print(
    complaint_analysis["complaint_category"]
    .value_counts()
)

COMPLAINT DATASET CREATED
-------------------------
Reviews: 300
Rows after category expansion: 426

Potential complaints: 217
No keyword match: 83

Category counts:
complaint_category
Product Not Received       102
No Clear Complaint          83
Seller / Service Issue      51
Late Delivery               32
Product Damage / Defect     27
Cancellation / Refund       25
Missing Product / Item      23
Shipping / Logistics        22
Payment                     22
Product Quality             18
Wrong Product               11
Packaging                   10
Name: count, dtype: int64
